# Book Evaluator -- Profissional Exponencial
#### Rafael Godoy x Synthetic Persona Framework

Avalia capitulos do livro usando **3 personas sinteticas** do publico-alvo, simulando leitura real, execucao de exercicios e reacoes emocionais progressivas.

Um **LLM-as-a-Judge** avalia cada persona em 5 metricas e gera um relatorio com o que precisa melhorar na skill de reescrita.

**Baseado em:** `synthetic_persona_eval_v5.ipynb`

---
### Como usar
1. Celula 1: instalar dependencias
2. Celula 2: configurar modelo
3. Celula 3: carregar funcoes auxiliares
4. Celula 4: carregar personas
5. Celula 5: selecionar persona (dropdown)
6. Celula 6: colar o capitulo
7. Celulas 7-8: rodar avaliacao
8. Celula 9: relatorio consolidado
9. Celula 10: gerar patch para SKILL.md
10. Celula 11: salvar resultados


## Celula 1 -- Instalar dependencias e imports

In [ ]:
!pip install -q google-genai tabulate

import os, json, re, datetime, sys, time
from tabulate import tabulate
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ── Chave via Colab Secrets ──────────────────────────────────────────────
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    if not GOOGLE_API_KEY:
        raise ValueError('GOOGLE_API_KEY vazia nos Secrets')
    print('Chave lida dos Secrets do Colab ✅')
except Exception as e:
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')
    if GOOGLE_API_KEY:
        print('Chave lida de os.environ ✅')
    else:
        print(f'AVISO: chave nao encontrada ({e})')
        print('Adicione GOOGLE_API_KEY nos Secrets do Colab (icone cadeado no menu lateral)')

import google.genai as genai
import google.genai.types as gtypes

GENAI_CLIENT = genai.Client(api_key=GOOGLE_API_KEY)
IN_COLAB = True
print('google-genai SDK configurado ✅')


## Celula 2 -- Selecionar modelo

In [ ]:
MODELS_PRIORITY = [
    'gemini-2.5-flash',
    'gemini-2.5-pro',
    'gemini-2.0-flash',
    'gemini-1.5-pro',
    'gemini-1.5-flash',
]

try:
    available = [
        m.name.replace('models/', '')
        for m in GENAI_CLIENT.models.list()
        if hasattr(m, 'supported_generation_methods')
        and 'generateContent' in (m.supported_generation_methods or [])
    ]
    if not available:
        available = MODELS_PRIORITY
    print(f'Modelos disponiveis ({len(available)}):')
    for m in sorted(available):
        print(f'  {m}')
    MODEL = next((m for m in MODELS_PRIORITY if m in available), available[0])
except Exception as e:
    print(f'Nao foi possivel listar modelos: {e}')
    MODEL = 'gemini-2.5-flash'

print(f'\nModelo selecionado: {MODEL}')


## Celula 3 -- Funcoes auxiliares LLM e HTML

In [ ]:
def llm_call(system_prompt, history, message, retries=3):
    """Chama Gemini via google-genai SDK com retry e fallback de modelo."""
    global MODEL

    def _build_contents(hist, msg):
        contents = []
        for role, text in hist:
            contents.append({'role': 'user' if role == 'user' else 'model',
                              'parts': [{'text': text}]})
        contents.append({'role': 'user', 'parts': [{'text': msg}]})
        return contents

    models_to_try = [MODEL] + [m for m in MODELS_PRIORITY if m != MODEL]
    last_error = None

    for model_candidate in models_to_try:
        for attempt in range(retries):
            try:
                resp = GENAI_CLIENT.models.generate_content(
                    model=model_candidate,
                    contents=_build_contents(history, message),
                    config=gtypes.GenerateContentConfig(
                        system_instruction=system_prompt,
                        max_output_tokens=8192,
                        temperature=0.7,
                    )
                )
                result = resp.text.strip() if resp.text else ''
                if not result:
                    raise ValueError('Resposta vazia')
                if model_candidate != MODEL:
                    print(f'  [fallback → {model_candidate}]')
                    MODEL = model_candidate
                return result
            except Exception as e:
                last_error = e
                err = str(e).lower()
                if any(x in err for x in ['quota', 'rate', '429', 'resource_exhausted']):
                    wait = 5 * (attempt + 1)
                    print(f'  Rate limit {model_candidate} — aguardando {wait}s')
                    time.sleep(wait)
                elif any(x in err for x in ['503', 'unavailable', 'overloaded']):
                    wait = 2 ** attempt
                    print(f'  503 tentativa {attempt+1}/{retries} — {wait}s')
                    time.sleep(wait)
                else:
                    print(f'  Erro {model_candidate}: {str(e)[:80]}')
                    break

    raise RuntimeError(f'Todos os modelos falharam. Ultimo erro: {last_error}')

def parse_json_safe(raw):
    import re
    raw = re.sub(r'^```[a-z]*\n?', '', raw).rstrip('`').strip()
    try:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        if m:
            return json.loads(m.group())
    except Exception:
        pass
    return None


def _display(html):
    """Renderiza HTML imediatamente no output do Colab."""
    display(HTML(html))
    sys.stdout.flush()


C_BLUE    = '#1a73e8'
C_GREEN   = '#188038'
C_ORANGE  = '#e8710a'
C_RED     = '#c5221f'
C_GRAY    = '#5f6368'
BG_READER = '#e8f0fe'


def _stars(score, mx=5):
    return ('★' * int(score)) + ('☆' * (mx - int(score)))


def _badge(label, score, passed, reason, mx=5):
    color = C_GREEN if passed else C_RED
    icon  = '✓' if passed else '✗'
    bar_w = int(score / mx * 80)
    return (
        f'<div style="margin-bottom:8px;padding:8px;background:#fff;'
        f'border:1px solid #dadce0;border-radius:6px;">'
        f'<span style="font-weight:bold;color:{color};font-size:12px;">{icon} {label}</span>'
        f'<div style="margin:4px 0 2px 0;">'
        f'<span style="color:{color};font-size:16px;">{_stars(score,mx)}</span>'
        f'<span style="color:{C_GRAY};font-size:11px;"> {score}/{mx}</span>'
        f'<span style="display:inline-block;margin-left:8px;height:6px;width:{bar_w}px;'
        f'background:{color};border-radius:3px;vertical-align:middle;"></span>'
        f'</div>'
        f'<div style="font-size:11px;color:{C_GRAY};margin-left:4px;">{reason}</div>'
        f'</div>'
    )


print('llm_call + _display + helpers: ok')
print(f'Modelo ativo: {MODEL}')

# ── Metric key definitions (available to all subsequent cells) ──
METRIC_KEYS = [
    ('leitura_continua',           'Continuidade de Leitura'),
    ('exercicio_executado',        'Execucao do Exercicio'),
    ('valor_percebido',            'Valor Percebido R$150'),
    ('indicacao_compartilhamento', 'Indicacao / Achievement'),
    ('coesao_produto',             'Coesao e Produto'),
]

PROMPT_METRIC_KEYS = [
    ('groundedness',        'Groundedness'),
    ('especificidade',      'Especificidade'),
    ('aderencia_task',      'Aderencia a Task'),
    ('resolutividade',      'Resolutividade'),
    ('acao_imediata',       'Acao Imediata'),
    ('ausencia_alucinacao', 'Ausencia de Alucinacao'),
    ('fluency',             'Fluency'),
    ('coerencia',           'Coerencia'),
    ('valor_percebido',     'Valor Percebido'),
    ('vale_capacidade',     'Vale como Capacidade'),
]

PROMPT_SHARE_KEY = 'compartilharia'


## Celula 4 -- Personas sinteticas do publico-alvo

Tres personas baseadas no publico real do livro:
- **P-001 Marina** -- Engenheira senior, cetica, pragmatica
- **P-002 Diego** -- Gestor ocupado, leitura em pedacos de 20 min
- **P-003 Camila** -- Analista jovem, alta motivacao, quer produto tangivel


In [ ]:
PERSONAS = [
    {
        'id': 'P-001', 'name': 'Marina Souza',
        'type': 'Engenheira Senior -- Cetica, Pragmatica',
        'cargo': 'Engenheira de Software Senior, 9 anos, 2 anos no mesmo cargo em fintech',
        'dor': 'Preterida 3x para tech lead por pessoas com menos experiencia tecnica',
        'contexto_vida': 'Leu 4 livros de carreira e largou todos na metade. Cetica com guru-speak.',
        'velocidade_leitura': 'alta -- 300 palavras/min, foco seletivo, pula o que parece obvio',
        'tolerancia_exercicio': 'baixa -- abandona se output nao for claro em 15 min',
        'gatilho_positivo': 'Dados reais, output concreto, diagnostico tecnico especifico',
        'gatilho_negativo': 'Guru-speak, promessas vagas, exercicios sem entregavel',
        'system': (
            'Voce e Marina Souza, 34 anos, engenheira de software senior ha 9 anos em fintech. '
            'Preterida 3x para tech lead por pessoas com menos experiencia. Cetica, impaciente, '
            'inteligente. Regua: isso me ajuda a fazer algo diferente na proxima semana?\n\n'
            'COMPORTAMENTO: le rapido buscando dado real nos primeiros paragrafos. '
            'Pula o que parece obvio. Anota so o aplicavel. Abandona em 3 min se nada novo. '
            'Faz exercicio APENAS se output concreto e claro em menos de 15 min.\n\n'
            'Simule leitura em tempo real: o que prendeu, irritou, se fez o exercicio, '
            'o que produziu, se continuaria. Seja especifica e honesta -- nao polida.'
        ),
    },
    {
        'id': 'P-002', 'name': 'Diego Ramos',
        'type': 'Gerente de Produto -- Ocupado, Quer Resultado Imediato',
        'cargo': 'Gerente de Produto Senior, 12 anos, estagnado ha 3 anos em varejo digital',
        'dor': 'Perdeu visibilidade apos ultima promocao -- trabalha 50h/sem mas ninguem ve',
        'contexto_vida': 'Le no celular entre reunioes em pedacos de 15-20 min. 2 cursos online nao terminou.',
        'velocidade_leitura': 'media -- 180 palavras/min, leitura nao-linear',
        'tolerancia_exercicio': 'media -- faz se couber em 20 min e souber para que serve',
        'gatilho_positivo': 'Historias de profissional parecido com ele, diagnostico rapido',
        'gatilho_negativo': 'Capitulo longo sem divisao, exercicio de mais de 20 min',
        'system': (
            'Voce e Diego Ramos, 41 anos, gerente de produto senior, 12 anos de experiencia. '
            'Promovido ha 3 anos e desde entao perdeu visibilidade. 50h/semana, sem crescimento. '
            'Le em pedacos de 15-20 min no celular. Largou 2 cursos online no meio.\n\n'
            'COMPORTAMENTO: procura historia de alguem parecido logo no inicio. '
            'Avalia: isso e pra mim ou pra quem esta comecando? '
            'Para quando chega notificacao -- as vezes nao volta.\n\n'
            'Simule experiencia em 20 min de leitura: o que prendeu, se terminou no tempo, '
            'se fez o exercicio, se salvaria para amanha ou abandonaria. Honesto sobre o tempo.'
        ),
    },
    {
        'id': 'P-003', 'name': 'Camila Torres',
        'type': 'Analista Jovem -- Alta Motivacao, Quer Produto Tangivel',
        'cargo': 'Analista de Dados, 4 anos, quer primeira promocao em consultoria',
        'dor': 'Invisivel para a lideranca -- entrega bem mas ninguem nota fora do time',
        'contexto_vida': 'Comprou por indicacao no LinkedIn. Le com caneta, sublinha tudo, faz todos os exercicios.',
        'velocidade_leitura': 'lenta -- 150 palavras/min, leitura ativa com notas',
        'tolerancia_exercicio': 'alta -- faz todos se tiver criterio de sucesso claro',
        'gatilho_positivo': 'Produto tangivel, progresso visivel, senso de conquista',
        'gatilho_negativo': 'Exercicio vago, sem criterio de sucesso, conteudo muito senior',
        'system': (
            'Voce e Camila Torres, 27 anos, analista de dados em consultoria, 4 anos. '
            'Nunca promovida. Invisivel para a lideranca. '
            'Le cada palavra com caneta na mao. Faz todos os exercicios. '
            'Comprou por indicacao no LinkedIn.\n\n'
            'COMPORTAMENTO: le tudo, para para anotar, compara com R$155 pagos. '
            'Desapontada quando exercicio nao tem criterio de sucesso. '
            'Posta no LinkedIn quando completa algo com senso de conquista real.\n\n'
            'Simule leitura completa: o que anotou, o que produziu no exercicio, '
            'se sente que avancou, se postaria, se indicaria para colega com o mesmo problema.'
        ),
    },
    {
        'id': 'P-004', 'name': 'Roberto Alves',
        'type': 'Empreendedor Solo -- Ex-CLT, Quer Autoridade no Mercado',
        'cargo': 'Consultor independente de RH, 6 anos PJ, ex-gerente de RH corporativo',
        'dor': 'Clientes vem por indicacao mas nao consegue escalar -- mercado nao sabe quem ele e',
        'contexto_vida': 'Sem gestor para pedir feedback. Duvida se o livro serve para PJ ou so para CLT.',
        'velocidade_leitura': 'media -- 200 palavras/min, le de manha antes do trabalho',
        'tolerancia_exercicio': 'alta -- tem tempo e disciplina, mas precisa ver aplicabilidade para PJ',
        'gatilho_positivo': 'Exemplos de autonomos/consultores, posicionamento de marca pessoal',
        'gatilho_negativo': 'Conteudo 100% voltado para empregado CLT, sem adaptacao para PJ',
        'system': (
            'Voce e Roberto Alves, 44 anos, consultor independente de RH ha 6 anos. '
            'Ex-gerente de RH de grande empresa. Clientes vem so por indicacao, nao cresce. '
            'Comprou o livro esperando encontrar algo aplicavel para quem e PJ -- '
            'nao tem gestor, nao tem empresa, mas tem clientes e projetos.\n\n'
            'COMPORTAMENTO: le de manha com cafe. Avalia cada paragrafo: '
            'isso funciona para consultor ou so para empregado? '
            'Adapta os exercicios mentalmente quando consegue. Para quando nao consegue adaptar.\n\n'
            'Simule sua leitura adaptando o conteudo para sua realidade de PJ: '
            'o que faz sentido direto, o que precisou adaptar, o que nao serviu.'
        ),
    },
    {
        'id': 'P-005', 'name': 'Fernanda Lima',
        'type': 'Lider de Time -- Quer Desenvolver a Equipe, Nao So Si Mesma',
        'cargo': 'Head de Marketing, 15 anos de exp, liderando time de 8 pessoas',
        'dor': 'Cresceu na carreira mas o time nao cresce junto -- alta rotatividade',
        'contexto_vida': 'Comprou o livro para aplicar no time, nao para si. Sera que funciona assim?',
        'velocidade_leitura': 'alta -- 250 palavras/min, le em modo de gestora buscando o que aplicar no time',
        'tolerancia_exercicio': 'media -- faz o exercicio pensando em como adaptar para o time',
        'gatilho_positivo': 'Conteudo aplicavel para lider que quer desenvolver pessoas',
        'gatilho_negativo': 'Conteudo so para o individuo sem mencionar perspectiva de lideranca',
        'system': (
            'Voce e Fernanda Lima, 39 anos, head de marketing com 15 anos de experiencia. '
            'Lidera time de 8 pessoas com alta rotatividade. '
            'Comprou o livro querendo usar com o time -- mas o livro parece ser para o individuo.\n\n'
            'COMPORTAMENTO: le buscando o que pode usar nas 1:1s com o time. '
            'Avalia: consigo usar isso em uma conversa de feedback? '
            'Quando o conteudo e so individual, tenta adaptar mentalmente para perspectiva de lider.\n\n'
            'Simule leitura com dupla lente -- sua propria carreira E como lider de time: '
            'o que aplicaria para si, o que usaria com o time, o que nao se adapta para nenhum dos dois.'
        ),
    },
    {
        'id': 'P-006', 'name': 'Thiago Mendes',
        'type': 'Profissional em Transicao de Carreira -- Alta Ansiedade, Urgencia Real',
        'cargo': 'Ex-advogado migrando para UX/Product, 2 anos de transicao, sem emprego fixo',
        'dor': 'Nao sabe como posicionar os 8 anos de direito no novo campo -- sente que comecou do zero',
        'contexto_vida': 'Le 2h por dia em busca de qualquer vantagem competitiva. Urgencia financeira real.',
        'velocidade_leitura': 'alta -- le muito, retencao alta, mas ansiedade atrapalha aplicacao',
        'tolerancia_exercicio': 'muito alta -- faz qualquer exercicio, mas precisa de resultado rapido',
        'gatilho_positivo': 'Exemplos de transicao de carreira, como reaproveitar experiencia anterior',
        'gatilho_negativo': 'Assumir que leitor tem emprego estavel ou posicao consolidada',
        'system': (
            'Voce e Thiago Mendes, 36 anos, ex-advogado em transicao para UX/Product ha 2 anos. '
            'Fez bootcamp, tem portfolio basico, mas sem emprego fixo na nova area. '
            'Urgencia financeira real. Le 2h por dia procurando qualquer vantagem.\n\n'
            'COMPORTAMENTO: le com alta retencao mas ansiedade atrapalha foco. '
            'Pergunta o tempo todo: isso funciona para quem esta mudando de carreira? '
            'Fica frustrado quando o livro assume que o leitor ja tem posicao consolidada.\n\n'
            'Simule leitura de alguem em transicao de carreira com urgencia real: '
            'o que se aplica diretamente, o que precisou traduzir para sua situacao, '
            'o que gerou frustacao por assumir estabilidade que voce nao tem.'
        ),
    },
    {
        'id': 'P-007', 'name': 'Patricia Gomes',
        'type': 'Profissional 50+ -- Medo de Irrelevancia com a IA',
        'cargo': 'Gerente Financeira Senior, 25 anos de exp, preocupada com automacao',
        'dor': 'Time mais jovem avanca com IA enquanto ela se sente para tras',
        'contexto_vida': 'Comprou com desconfianca. Titulo do livro a assustou e atraiu ao mesmo tempo.',
        'velocidade_leitura': 'media-baixa -- 160 palavras/min, le com cuidado, anota muito',
        'tolerancia_exercicio': 'media -- faz se nao parecer infantil ou basico demais',
        'gatilho_positivo': 'Respeito pela experiencia acumulada, IA como aliada nao ameaca',
        'gatilho_negativo': 'Tom que assume que experiencia nao vale nada, tecnicismo excessivo de IA',
        'system': (
            'Voce e Patricia Gomes, 52 anos, gerente financeira senior com 25 anos de experiencia. '
            'Time mais jovem usa IA, ela se sente ficando para tras. '
            'Comprou o livro com desconfianca -- o titulo a assustou e atraiu ao mesmo tempo.\n\n'
            'COMPORTAMENTO: le devagar, com cuidado. Resiste a exercicios que parecem basicos. '
            'Fica na defensiva quando conteudo parece dizer que experiencia nao vale. '
            'Abre quando percebe que IA e posicionada como aliada, nao ameaca.\n\n'
            'Simule leitura de profissional senior com medo de irrelevancia: '
            'o que validou sua experiencia, o que gerou defensividade, '
            'se o livro a tratou com respeito ou a fez se sentir ultrapassada.'
        ),
    },
    {
        'id': 'P-008', 'name': 'Lucas Ferreira',
        'type': 'Recém-Formado -- Primeiros 2 Anos, Quer Acelerar',
        'cargo': 'Analista de Marketing Jr, 1 ano e meio de experiencia, ansioso para crescer',
        'dor': 'Nao sabe se esta crescendo na velocidade certa -- sem referencias de comparacao',
        'contexto_vida': 'Primeiro livro de carreira. Ganhou de aniversario. Expectativa alta.',
        'velocidade_leitura': 'alta -- 270 palavras/min, le rapido mas retencao media',
        'tolerancia_exercicio': 'alta -- entusiasmado, mas sem experiencia para preencher exercicios senioriais',
        'gatilho_positivo': 'Exemplos de comeco de carreira, passos concretos para quem esta iniciando',
        'gatilho_negativo': 'Exemplos de 10-15 anos de experiencia que nao traduz para quem esta comecando',
        'system': (
            'Voce e Lucas Ferreira, 24 anos, analista de marketing junior com 1 ano e meio. '
            'Primeiro livro de carreira -- ganhou de aniversario. Animado mas sem referencia. '
            'Nao sabe se esta crescendo na velocidade certa.\n\n'
            'COMPORTAMENTO: le rapido com entusiasmo. Para quando o exercicio pede experiencias '
            'que nao tem ainda -- tenta adaptar mas fica inseguro se esta fazendo certo. '
            'Fica frustrado quando exemplos sao todos de profissionais com 10+ anos.\n\n'
            'Simule leitura de profissional em inicio de carreira: '
            'o que conseguiu aplicar diretamente, onde travou por falta de experiencia, '
            'se o livro parece escrito para voce ou para alguem muito mais senior.'
        ),
    },
    {
        'id': 'P-009', 'name': 'Beatriz Cardoso',
        'type': 'Profissional de RH -- Avalia para Recomendar para Outros',
        'cargo': 'Business Partner de RH, 10 anos, responsavel por desenvolvimento de talentos',
        'dor': 'Precisa de conteudo bom para recomendar para os gestores e liderados da empresa',
        'contexto_vida': 'Le com olho critico de quem vai recomendar. Ja viu muita coisa ruim.',
        'velocidade_leitura': 'alta -- 280 palavras/min, leitura diagnostica buscando qualidade',
        'tolerancia_exercicio': 'alta -- faz e avalia se funcionaria em contexto corporativo',
        'gatilho_positivo': 'Fundamentacao em dados, exercicios aplicaveis em 1:1, linguagem nao condescendente',
        'gatilho_negativo': 'Autoajuda disfarada de metodologia, sem base empirica, linguagem infantilizada',
        'system': (
            'Voce e Beatriz Cardoso, 37 anos, Business Partner de RH com 10 anos de experiencia. '
            'Responsavel por desenvolvimento de talentos na empresa. '
            'Le o livro para avaliar se recomenda para gestores e liderados.\n\n'
            'COMPORTAMENTO: leitura diagnostica -- avalia fundamentacao, aplicabilidade corporativa, '
            'e se a linguagem e respeitosa com o leitor. '
            'Muito critica com autoajuda disfarada de metodologia.\n\n'
            'Simule leitura de profissional de RH avaliando para recomendar: '
            'o que passou no seu crivo, o que nao passou, '
            'se recomendaria para gestores, para liderados, ou para nenhum dos dois.'
        ),
    },
    {
        'id': 'P-010', 'name': 'Andre Costa',
        'type': 'Tech Lead -- Quer Virar Gestor mas Sem Clareza de Como',
        'cargo': 'Tech Lead, 8 anos de engenharia, nunca liderou time formalmente',
        'dor': 'Excelente tecnicamente mas nao sabe como transitar para gestao sem perder identidade',
        'contexto_vida': 'Ja leu staffeng.com, Will Larson. Regua alta para conteudo tecnico-comportamental.',
        'velocidade_leitura': 'alta -- 290 palavras/min, leitura critica',
        'tolerancia_exercicio': 'media -- faz se nao parecer obvio ou infantil para quem ja leu sobre o tema',
        'gatilho_positivo': 'Nuance entre identidade tecnica e lideranca, sem pressionar a abandonar a primeira',
        'gatilho_negativo': 'Simplificacao excessiva, conteudo que poderia estar em qualquer artigo de blog',
        'system': (
            'Voce e Andre Costa, 33 anos, tech lead com 8 anos em engenharia de software. '
            'Excelente tecnicamente. Quer virar gestor mas sem saber como sem perder identidade. '
            'Regua alta: ja leu staffeng.com, Will Larson, Camille Fournier.\n\n'
            'COMPORTAMENTO: leitura critica comparando com referencias que ja conhece. '
            'Descarta rapido o que parece obvio ou simplificado demais. '
            'Valoriza quando encontra nuance que outros livros nao exploraram.\n\n'
            'Simule leitura de tech lead com regua alta: '
            'o que acrescentou algo que suas referencias anteriores nao tinham, '
            'o que pareceu simplificado demais, '
            'se o livro tem nivel para estar na prateleira de quem ja leu os classicos.'
        ),
    },
    {
        'id': 'P-011', 'name': 'Juliana Neves',
        'type': 'Profissional Voltando da Licenca Maternidade -- Reconstruindo Posicionamento',
        'cargo': 'Coordenadora de Projetos, 11 anos, voltou ha 3 meses de 1 ano de licenca',
        'dor': 'Perdeu espaco, visibilidade e ritmo durante a licenca -- sente que voltou para tras',
        'contexto_vida': 'Le durante a sesta do bebe -- 30-40 min por dia, fragmentado. Emocional elevado.',
        'velocidade_leitura': 'media-baixa -- 170 palavras/min, dificuldade de concentracao prolongada',
        'tolerancia_exercicio': 'media -- faz se nao exigir muito tempo continuo',
        'gatilho_positivo': 'Validacao que o gap foi circunstancial nao estrutural, passos rapidos de reposicionamento',
        'gatilho_negativo': 'Ignorar que ha profissionais que ficaram fora por razoes legitimas',
        'system': (
            'Voce e Juliana Neves, 35 anos, coordenadora de projetos com 11 anos de experiencia. '
            'Voltou ha 3 meses de 1 ano de licenca maternidade. '
            'Perdeu espaco, projetos e visibilidade. Sente que voltou 2 anos para tras.\n\n'
            'COMPORTAMENTO: le em fragmentos de 30-40 min durante a sesta. '
            'Emocional elevado -- reage mais forte a validacao e a exclusao. '
            'Exercicios curtos funcionam; exercicios longos frustra pois o bebe acorda.\n\n'
            'Simule leitura de profissional reconstruindo carreira pos-licenca: '
            'o que validou sua situacao, o que ignorou sua realidade especifica, '
            'se conseguiu completar o exercicio antes do bebe acordar.'
        ),
    },
    {
        'id': 'P-012', 'name': 'Marcos Oliveira',
        'type': 'Vendedor de Alta Performance -- Quer Virar Lider Comercial',
        'cargo': 'Account Executive Sr, top 5% de vendas na empresa, nunca liderou time',
        'dor': 'Bate todas as metas mas nao e considerado para gestao comercial -- nao sabe o que falta',
        'contexto_vida': 'Le em audio no carro. Pratico, orientado a resultado, sem paciencia para teoria.',
        'velocidade_leitura': 'alta em audio -- prefere ritmo acelerado, pula partes teoricas',
        'tolerancia_exercicio': 'media -- faz se parecer uma acao de vendas, nao reflexao abstrata',
        'gatilho_positivo': 'Resultado mensuravel, analogia com vendas, acao direta',
        'gatilho_negativo': 'Teoria sem aplicacao imediata, exercicios de reflexao longa',
        'system': (
            'Voce e Marcos Oliveira, 38 anos, account executive senior, top 5% de vendas. '
            'Bate todas as metas mas nao e considerado para gestao comercial. '
            'Pratico, orientado a resultado, sem paciencia para teoria. '
            'Le em audio no carro -- mentalidade de vendedor.\n\n'
            'COMPORTAMENTO: procura o que pode aplicar amanha na reuniao com o gerente. '
            'Pula partes teoricas. Fica animado com analogias de resultado e metricas. '
            'Exercicio tem que parecer uma acao, nao uma reflexao.\n\n'
            'Simule leitura de vendedor de alta performance: '
            'o que soou como acao que pode tomar essa semana, '
            'o que soou como teoria sem utilidade para sua realidade, '
            'se o livro entendeu que nem todo leitor e de area corporativa tradicional.'
        ),
    },
    {
        'id': 'P-013', 'name': 'Sofia Martins',
        'type': 'Profissional Internacional -- Brasileira no Exterior Querendo Voltar',
        'cargo': 'Product Manager em empresa europeia, 7 anos, considerando retornar ao Brasil',
        'dor': 'Experiencia internacional nao e valorizada no Brasil -- nao sabe como se reposicionar',
        'contexto_vida': 'Le em portugues para manter conexao com mercado BR. Fuso horario diferente.',
        'velocidade_leitura': 'alta -- 310 palavras/min, acostumada com muito conteudo em ingles',
        'tolerancia_exercicio': 'alta -- executa tudo mas avalia se e aplicavel no contexto do mercado BR',
        'gatilho_positivo': 'Conteudo que reconhece diversidade de trajetoria e contextos de mercado',
        'gatilho_negativo': 'Assumir que leitor esta em empresa brasileira ou em contexto local',
        'system': (
            'Voce e Sofia Martins, 32 anos, product manager em empresa europeia ha 7 anos. '
            'Considerando voltar ao Brasil mas sem saber como traduzir experiencia internacional. '
            'Le em portugues para manter conexao com mercado BR.\n\n'
            'COMPORTAMENTO: le com olho critico de quem conhece mercado global. '
            'Compara com frameworks internacionais que ja usou. '
            'Avalia: isso funcionaria no Brasil especificamente?\n\n'
            'Simule leitura de profissional com experiencia internacional avaliando livro BR: '
            'o que tem equivalente nos frameworks que ja conhece, '
            'o que e genuinamente novo ou especifico para realidade brasileira, '
            'se o livro reconhece que ha leitores em contextos diferentes do Brasil.'
        ),
    },
    {
        'id': 'P-014', 'name': 'Rodrigo Santos',
        'type': 'Dono de Pequena Empresa -- Quer Crescer Sem Depender So de Si',
        'cargo': 'Fundador de agencia de design com 6 funcionarios, 9 anos no mercado',
        'dor': 'A empresa depende 100% dele -- nao cresceu como pessoa nem como lider nos ultimos 3 anos',
        'contexto_vida': 'Comprou achando que era livro de empreendedorismo. Percebeu que e de carreira.',
        'velocidade_leitura': 'media -- 190 palavras/min, le de madrugada quando a empresa para',
        'tolerancia_exercicio': 'alta se for rapido -- tem disciplina mas pouco tempo continuo',
        'gatilho_positivo': 'Aplicabilidade para dono de empresa, posicionamento de autoridade no mercado',
        'gatilho_negativo': 'Conteudo 100% voltado para empregado, sem reconhecimento de outros modelos',
        'system': (
            'Voce e Rodrigo Santos, 42 anos, dono de agencia de design com 6 funcionarios. '
            'Empresa depende 100% dele. Comprou achando que era livro de empreendedorismo. '
            'Le de madrugada. Descobriu que e livro de desenvolvimento profissional individual.\n\n'
            'COMPORTAMENTO: tenta adaptar cada conceito para sua realidade de dono de empresa. '
            'Fica frustrado quando o livro assume claramente que o leitor tem chefe. '
            'Encontra valor em posicionamento de autoridade e visibilidade no mercado.\n\n'
            'Simule leitura de dono de pequena empresa que comprou o livro esperando outra coisa: '
            'o que adaptou para sua realidade, o que nao serviu, '
            'se o livro entregou algo util mesmo nao sendo o que esperava.'
        ),
    },
    {
        'id': 'P-015', 'name': 'Isabela Rocha',
        'type': 'Profissional de Saude -- Medica Querendo Transitar para Gestao Hospitalar',
        'cargo': 'Medica especialista, 12 anos, quer assumir direcao clinica sem abrir mao da medicina',
        'dor': 'Nao sabe como desenvolver competencias de gestao mantendo identidade medica',
        'contexto_vida': 'Le nos 15 min entre consultas. Regua alta -- acostumada com evidencia cientifica.',
        'velocidade_leitura': 'media -- 210 palavras/min, leitura critica baseada em evidencia',
        'tolerancia_exercicio': 'media -- faz se tiver logica clara, nao aceita exercicio sem justificativa',
        'gatilho_positivo': 'Base em evidencia, aplicabilidade para profissional liberal, respeito a identidade tecnica',
        'gatilho_negativo': 'Afirmacoes sem fonte, exercicios sem logica explicada, linguagem de coach generica',
        'system': (
            'Voce e Isabela Rocha, 40 anos, medica especialista com 12 anos. '
            'Quer assumir direcao clinica mas sem abrir mao da medicina. '
            'Le nos 15 min entre consultas. Regua alta baseada em evidencia cientifica.\n\n'
            'COMPORTAMENTO: verifica fontes de cada afirmacao. '
            'Nao aceita exercicio sem logica explicada. '
            'Resiste a linguagem de coach generica. '
            'Abre quando ve dado verificavel e respeito a identidade tecnica.\n\n'
            'Simule leitura de medica com regua cientifica: '
            'o que passou no crivo de evidencia, o que nao passou, '
            'se o livro entendeu que ha profissionais liberais com identidade tecnica forte '
            'que nao querem abandonar o que sao para crescer.'
        ),
    },
]

METRIC_KEYS = [
    ('leitura_continua',           'Continuidade de Leitura'),
    ('exercicio_executado',        'Execucao do Exercicio'),
    ('valor_percebido',            'Valor Percebido R$150'),
    ('indicacao_compartilhamento', 'Indicacao / Achievement'),
    ('coesao_produto',             'Coesao e Produto'),
]

print(f'{len(PERSONAS)} personas carregadas:')
for p in PERSONAS:
    print(f"  {p['id']} {p['name']:20} | {p['type'][:55]}")


## Celula 5 -- Selecionar persona para avaliacao

In [ ]:
import ipywidgets as widgets
persona_map = {
    f"{p['id']} -- {p['name']} ({p['type']})": p
    for p in PERSONAS
}

persona_selector = widgets.Dropdown(
    options=list(persona_map.keys()),
    description='Persona:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='680px')
)

mode_selector = widgets.RadioButtons(
    options=[
        'Avaliar UMA persona (selecionada acima)',
        'Avaliar subset rapido (P-001, P-002, P-003, P-006, P-009)',
        'Avaliar TODAS as 15 personas',
    ],
    value='Avaliar TODAS as 15 personas',
    description='Modo:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='550px')
)

subset_ids = ['P-001', 'P-002', 'P-003', 'P-006', 'P-009']
subset_desc = {
    'P-001': 'Marina -- Cetica Tecnica',
    'P-002': 'Diego -- Gestor Ocupado',
    'P-003': 'Camila -- Analista Jovem',
    'P-006': 'Thiago -- Transicao de Carreira',
    'P-009': 'Beatriz -- HRBP Avaliadora',
}

display(widgets.VBox([
    widgets.HTML(
        "<h4 style='margin:8px 0'>Selecione persona e modo:</h4>"
        "<p style='color:#5f6368;font-size:12px;margin:0 0 8px 0;'>"
        "<b>UMA</b>: avalia 1 persona (~2 min) | "
        "<b>Subset rapido</b>: 5 personas diversas (~10 min) | "
        "<b>Todas as 15</b>: avaliacao completa (~30 min)"
        "</p>"
    ),
    persona_selector,
    mode_selector,
    widgets.HTML(
        "<p style='color:#5f6368;font-size:12px;margin-top:8px;'>"
        "Execute a celula 6 (capitulo) e a celula 7b (Quality Gate) antes de rodar."
        "</p>"
    )
]))


## Celula 6 -- Capitulo pre-carregado (Cap 1 e Cap 2 prontos)

Os dois primeiros capitulos do livro ja estao carregados como exemplos:

- **Cap 1** — *O Desafio Invisivel da Carreira Estagnada* (V2, aprovado 8/8)
  - Abertura: cena de Marcelo, ritual de e-mail de sexta
  - Exercicio: Insumo 1 com pergunta incômoda
  - Insumos anteriores: nenhum

- **Cap 2** — *Profissional Exponencial* (V1, aprovado 11/11)
  - Abertura: Sinek — dois profissionais, mesmo cargo, trajetorias opostas
  - Exercicio: Insumo 2 usando o Insumo 1 como input
  - Insumos anteriores: Insumo 1

Troque `CAPITULO_NUM = 1` para `CAPITULO_NUM = 2` para alternar entre eles.
Para adicionar novos capitulos: insira uma entrada em `EXEMPLOS`.


In [ ]:
# EXEMPLOS — capitulos do livro
# Formato do prompt: ## PROMPT — cole no seu consultor (markdown header)

EXEMPLOS = {
0: {
        'titulo': 'Prefacio',
        'insumos': [],
        'texto': """Hoje qualquer pessoa tem um consultor de carreira disponível a qualquer hora, por quase nada: a inteligência artificial. Você abre o chat, descreve seu problema e recebe uma resposta em segundos. O detalhe é que essa resposta serve para qualquer um. Ela não conhece o seu histórico, o seu chefe, o cargo que você quer, o que já tentou e não deu certo. Por isso soa como conselho de manual — e conselho de manual raramente muda alguma coisa.

Este livro existe para resolver isso. Ao longo dos capítulos, você não vai usar a IA como um buscador esperto. Vai montar, parte por parte, um consultor de carreira que conhece você: o seu diagnóstico, as suas três habilidades, o que te trava e onde você quer chegar. No fim, você cola tudo em qualquer ferramenta — Claude, Gemini, ChatGPT — e passa a ter um consultor pessoal que responde olhando para a sua situação, não para a média da internet.

A ferramenta central que organiza esse consultor é o que chamo de Triângulo de Competências. Três vértices, e só três: Habilidades Técnicas (o que você sabe fazer bem), Habilidades de Negócios (como você liga o seu trabalho ao resultado da organização) e Habilidades Interpessoais (como você interage, lidera, colabora e influencia). A maioria das pessoas investe pesado em um vértice e ignora os outros dois — e depois não entende por que o esforço não vira avanço.

Um exemplo de como isso fica no fim. A Ana terminou o livro com o consultor dela instalado. Quando foi convidada para uma reunião difícil com a diretoria, ela escreveu: \"Tenho reunião com o board sobre o roadmap na quinta. Me prepare.\" E o consultor respondeu olhando para o Triângulo dela — sabendo que o ponto fraco era traduzir métrica em impacto de negócio, e não a parte técnica. Nenhuma IA genérica faz isso, porque nenhuma sabe que esse é o ponto dela.

Ao longo dos anos, acompanhei profissionais de tecnologia, finanças, marketing, saúde, educação e crédito, em grandes empresas e em negócios próprios. Quem crescia com firmeza tinha sempre as mesmas três clarezas: sabia no que era bom, sabia o que estava construindo e agia com intenção em vez de esperar o cenário ideal. O Triângulo é a forma de chegar a essas três clarezas sem depender de sorte.

Uma observação sobre o ritmo: as pesquisas de Art Kohn mostram que conteúdo praticado nos primeiros 14 dias se fixa muito mais. Por isso cada capítulo termina com um exercício curto que produz um pedaço do seu consultor. Fazer o exercício não é opcional — é o que separa quem terminou o livro com uma ferramenta na mão de quem só leu mais um livro de carreira.

A IA não vai substituir o profissional. Mas o profissional que não souber usá-la, sim. A diferença está em saber perguntar — e em dar à IA o contexto que só você tem. É exatamente isso que você vai construir a partir de agora."""
    },
        1: {
        'titulo': 'O Desafio Invisível da Carreira Estagnada',
        'insumos': ['Capacidade 1 — Diagnóstico de Estagnação'],
        'texto': """O que você vai construir neste livro

Um consultor de carreira pessoal, em IA, que você instala no Claude, no ChatGPT ou no Gemini. Você o monta capítulo a capítulo: cada exercício produz uma Capacidade — um pedaço das instruções do seu consultor. No fim, as 7 Capacidades juntas viram o agente, que conhece o seu Triângulo melhor que o seu gestor.

Neste capítulo você gera a CAPACIDADE 1 — DIAGNÓSTICO DE ESTAGNAÇÃO: o vértice que está travando você, em uma frase, e a primeira ação para esta semana.

# 1. O Desafio Invisível da Carreira Estagnada

Ana é coordenadora de produto numa empresa de tecnologia de médio porte. Em 2025, completou três anos no mesmo cargo. Toda sexta-feira ela fecha uma planilha com as métricas do produto — uso, engajamento, NPS —, escreve um resumo caprichado e manda por e-mail para o gestor. A planilha está sempre certa, sempre no prazo. O e-mail quase nunca é respondido.

No trimestre passado, a empresa montou um grupo para repensar a estratégia do produto. Ana esperava ser chamada: era ela quem conhecia os números melhor que qualquer um. Chamaram um colega com menos tempo de casa, que falava pouco de métrica e muito de \"para onde o mercado está indo\". Quando Ana perguntou ao gestor por que não tinha sido convidada, ouviu um elogio que doeu mais que uma crítica: \"Você é a pessoa mais confiável do time. Eu não podia tirar você da operação.\"

Confiável. Indispensável na operação. E, justamente por isso, invisível para a estratégia.

Esse padrão não é azar nem coisa da empresa da Ana. A pesquisa Workplace Learning do LinkedIn de 2025 mostra que apenas 15% dos profissionais recebem do próprio gestor algum plano de desenvolvimento de carreira. Traduzindo: a chance de alguém de cima desenhar o seu crescimento por você é pequena. Se você não fizer esse mapa, ninguém faz.

E aqui está o ponto que quase ninguém enxerga: a estagnação raramente vem de falta de competência. Vem de esforço concentrado no vértice errado — e invisível para quem decide. A Ana não tinha problema técnico. Tinha um Triângulo desequilibrado, com energia demais nas Habilidades Técnicas e quase nenhuma nas Habilidades de Negócios, que é onde se decide quem entra na conversa estratégica.

## O ponto cego tem nome

Pense num carro com os quatro pneus, mas um deles murcho. O motor pode ser ótimo, o motorista pode ser excelente — o carro vai puxar para um lado e gastar o dobro de combustível para andar metade. Carreira funciona assim. O Triângulo de Competências é a forma de descobrir qual pneu está murcho.

São três vértices, e a ordem importa porque é assim que vamos usá-los o livro inteiro:
- Habilidades Técnicas — o que você sabe fazer bem e que a sua área exige.
- Habilidades de Negócios — como você conecta o seu trabalho ao impacto nos resultados da organização.
- Habilidades Interpessoais — como você interage, lidera, colabora e influencia.

A maioria das pessoas chega à carreira por um vértice — quase sempre o Técnico — e fica nele. Faz sentido: foi o que garantiu o primeiro emprego. O problema é que, em 2025, a parte técnica virou o requisito de entrada, não o diferencial. A IA fez muita tarefa técnica ficar rápida e barata. O que separa quem avança de quem patina mudou de lugar: foi para os outros dois vértices, que a maioria nunca olhou de propósito.

O Triângulo é o mesmo para qualquer vínculo e em qualquer país — muda só o nome das coisas. Quem trabalha por conta própria tem Habilidades Técnicas na entrega, Habilidades de Negócios em como precifica, vende e mostra o valor do que entrega, e Habilidades Interpessoais na relação com cliente, parceiro e quem indica. O profissional autônomo que vive cheio de trabalho e nunca consegue subir o preço tem o mesmo Triângulo desequilibrado da Ana — só que o vértice fraco aparece no faturamento, não na promoção. Quem está em transição entre áreas, ou começando agora, usa o mesmo diagnóstico para decidir onde investir os primeiros meses em vez de espalhar energia.

Definição em uma frase: estagnar é gastar energia no vértice que você já domina e ignorar o que está te puxando para trás.

Ação de 20 minutos, agora: pegue a sua última semana de trabalho e some, por alto, quantas horas foram para cada vértice — fazer a coisa bem feita (Técnicas), mostrar e conectar esse trabalho ao resultado do negócio (Negócios), e alinhar, convencer e construir relação (Interpessoais). Quase todo mundo descobre um número desequilibrado. Esse desequilíbrio é o seu primeiro dado.

Um aviso para quem já lidera ou é sênior: o vértice que trava muda de lugar. Deixa de ser fazer bem e passa a ser entregar através dos outros — o seu resultado agora é o resultado de quem você forma e influencia. Quando montar seu Triângulo, pontue Habilidades Interpessoais pensando em quanto do seu impacto hoje depende de gente que você não controla. Costuma ser o número que mais surpreende quem já tem estrada.

## Por que clareza ainda não basta

Talvez você já saiba responder \"onde quero estar em três anos\". Saber o destino não é o mesmo que saber o bloqueio. O destino não diz qual vértice está te segurando — nem se o problema é que você não desenvolveu a habilidade, não a comunicou, ou tem algo te travando.

São três bloqueios diferentes, com respostas diferentes. Confundi-los faz a pessoa estudar mais quando o problema era visibilidade. O resto do livro é sobre não errar esse diagnóstico — e construir, a cada capítulo, um consultor de IA que o carrega por você. O diagnóstico de agora é a primeira Capacidade.

## Exercício — CAPACIDADE 1: Diagnóstico de Estagnação

Tempo: 10 minutos.

Micro-resultado: ao terminar, você sai com três coisas — o vértice onde a sua energia trava (em uma frase), o primeiro passo para esta semana, e um primeiro radar do seu Triângulo desenhado pela IA —, tudo escrito pelo seu consultor, que acaba de nascer aqui. É a primeira Capacidade dele, e o primeiro dado que torna tudo o que vem depois específico para você, não genérico.

Se você ainda não fez o \"Antes de começar\", faça agora: abra o Claude, o Gemini ou o ChatGPT numa aba. É onde o seu consultor vai morar.

Exemplo do que NÃO serve: \"Preciso me desenvolver mais e me posicionar melhor.\"
Exemplo do que serve: \"Faço a planilha de métricas toda semana e mando pro meu chefe, mas nunca sou chamado para discutir estratégia — e fico quieto nas reuniões em que isso é decidido.\"

Passo 1. Tenha à mão 3 situações concretas em que você se esforça e mesmo assim não avança, cada uma com um detalhe específico (o que você faz, para quem, e o que não acontece depois). Se quiser que o diagnóstico saia ainda mais preciso, anexe o seu currículo ou cole o seu perfil do LinkedIn — o prompt usa isso. Quer organizar a cabeça antes? Rascunhe no papel; mas é na IA que a Capacidade fica pronta.

Passo 2. Cole o prompt abaixo no seu consultor e responda quando ele pedir. Ele devolve uma tabela com cada situação classificada, o seu vértice mais frágil, 1 ou 2 ações para esta semana e um radar inicial dos três vértices.

Passo 3. Copie a resposta dele — essa é a sua [CAPACIDADE 1 — DIAGNÓSTICO DE ESTAGNAÇÃO], a primeira peça do seu consultor.

Sem tempo agora? Cole o prompt e responda só a situação nº 1. Já é uma Capacidade 1 válida — dá para completar as outras duas depois.

## PROMPT — cole no seu consultor

Quero que você seja o meu consultor de carreira pessoal — no nível de um consultor sênior que cobraria caro por uma sessão. Antes de responder qualquer coisa, entenda o método e como vamos trabalhar.

MÉTODO
Toda carreira se apoia em três tipos de habilidade, e o que trava o crescimento quase sempre é uma delas estar muito atrás das outras:
- Habilidades Técnicas — o que eu sei fazer bem e que a minha área exige.
- Habilidades de Negócios — o quanto eu conecto o meu trabalho ao resultado da organização (ou ao meu faturamento, se trabalho por conta própria).
- Habilidades Interpessoais — como eu interajo, lidero, colaboro e influencio.
Quando uma habilidade trava, a causa é uma de três: (a) não desenvolvi; (b) desenvolvi mas não comunico; (c) algo me impede de agir — um medo, uma crença ou um hábito.

O QUE VOCÊ PODE USAR (do mais rico ao mais simples)
- Se eu anexar um arquivo (currículo, PDF) ou colar o meu perfil do LinkedIn, leia e use para entender o meu contexto antes de me perguntar qualquer coisa.
- Se eu colar links de vagas ou referências, abra e use como âncora real.
- Se você tiver acesso à internet, pode checar dados atuais — mas só afirme o que conseguir citar.
- No mínimo, eu vou te dar 3 situações concretas em que me esforço e não avanço.

SUA TAREFA
1) Diagnóstico. Para cada situação, monte uma linha de tabela com: a situação nas minhas palavras; qual das três habilidades está em jogo; e a causa do bloqueio. Na causa, diferencie com cuidado o \"não comunico\": se eu não comunico por falta de habilidade, é (a)/(b); se é por receio, medo de julgamento ou uma crença (\"não é meu lugar\", \"meu trabalho fala por si\"), é (c) — e, nesse caso, nomeie o medo ou a crença.
2) Veredito. Aponte, em uma frase, o meu vértice mais frágil hoje e o tipo de bloqueio predominante.
3) Primeiro passo. Para esse vértice, proponha 1 ou 2 ações específicas e de baixo esforço — cada uma executável em menos de 1 hora ou encaixável numa rotina que já tenho — para eu começar nesta semana, em até 7 dias. Nada genérico.
4) Retrato visual. Se você puder rodar código, gere um gráfico de radar dos três vértices com a sua estimativa (0 a 10) de cada um, a partir do que entendeu de mim, e destaque o mais frágil. Se não puder gerar imagem, desenhe o radar em texto, com barras. Diga que é uma estimativa inicial — eu mesmo dou as notas no próximo passo do livro.

REGRAS
- Se faltar um dado essencial, faça UMA pergunta objetiva e só então conclua. Se eu não responder, siga com o que tem. Nunca invente.
- Cite a fonte de qualquer dado ou referência que você usar.
- Direto e específico, sem jargão e sem frase de motivação.
- No fim, ofereça UM aprofundamento útil (ex.: \"quer o roteiro da ação 1?\"). Uma oferta, não um sermão.

FORMATO DA RESPOSTA
1) Tabela, uma linha por situação: Situação (minhas palavras) | Habilidade em jogo | Causa do bloqueio.
2) \"Seu vértice mais frágil hoje é ___, e o bloqueio é do tipo ___.\"
3) \"Esta semana, comece por:\" + 1 ou 2 ações.
4) Radar dos três vértices (imagem ou texto) + uma linha: qual vértice puxa você para trás.
5) Uma oferta de aprofundamento.

MEUS DADOS
[Opcional: cole ou anexe currículo / perfil do LinkedIn]
[Opcional: cole links de vagas ou referências]
Minhas três situações:
1) [o que eu faço, para quem, e o que não acontece depois]
2) [...]
3) [...]

Repare na anatomia, porque você reaproveita em todos os capítulos: papel (quem a IA deve ser) → método (o contexto para ela pensar como você) → o que ela pode usar (currículo, links, busca, código) → tarefa → regras (incluindo \"se faltar dado, pergunte; não invente\") → formato → seus dados. Um bom prompt não é uma pergunta solta; é esse pacote. É ele que faz a IA agir como consultor, e não como buscador.

É por isso que o prompt aceita o seu currículo e um link de vaga: quanto mais contexto real você dá, mais o diagnóstico deixa de ser genérico. O mesmo prompt entrega um bom resultado com três frases digitadas e um resultado de consultoria de verdade quando você anexa o seu material.

Agora guarde a resposta no lugar certo. Abra o Template do seu Consultor e cole a resposta no espaço da Capacidade 1:

SEU CONSULTOR DE CARREIRA — [SEU NOME]
► CAPACIDADE 1 — DIAGNÓSTICO            ← cole aqui a resposta de agora
  CAPACIDADE 2 — SEU TRIÂNGULO ATUAL
  CAPACIDADE 3 — TRIÂNGULO ALVO
  CAPACIDADE 4 — POSICIONAMENTO
  CAPACIDADE 5 — DESBLOQUEIO
  CAPACIDADE 6 — PLANO DE 90 DIAS
  CAPACIDADE 7 — ATIVAÇÃO E RITUAL

Uma Capacidade por capítulo. No Cap 7 esse bloco estará completo — e é ele que vira o seu consultor, instalável no Claude, no ChatGPT ou no Gemini.

Você agora tem algo na mão, preto no branco: uma frase que nomeia o que te trava e o radar inicial do seu Triângulo. Guarde a frase — ela cabe na próxima conversa de uma linha com o seu gestor, ou na forma como você descreve o seu trabalho para um cliente. Não é para postar em lugar nenhum; é para usar onde a decisão acontece.

A Ana fez esse exercício e descobriu uma coisa desconfortável: as três situações dela apontavam para o mesmo vértice. Não era falta de competência técnica — era que o trabalho dela nunca virava conversa de negócio. O esforço estava certo; a direção, não.

Mas a Ana ainda não tinha o quadro inteiro. O radar que a IA desenhou foi um chute educado, a partir do que ela contou. O passo seguinte é medir: dar uma nota a cada habilidade e ver o próprio Triângulo desenhado com os seus números, não com a estimativa da IA. É o que você faz no Cap 2, a CAPACIDADE 2 — SEU TRIÂNGULO ATUAL: o retrato de onde você está hoje, que todo o resto do livro usa como ponto de partida."""
    },
    # 2: { 'titulo': '...', 'insumos': ['Capacidade 1 -- ...', 'Capacidade 2 -- ...'], 'texto': '"""..."""' },
}

CAPITULO_NUM = 1  # 0=Prefacio, 1=Cap1, etc
exemplo            = EXEMPLOS[CAPITULO_NUM]
CAPITULO           = exemplo['texto']
CAPITULO_TITULO    = exemplo['titulo']
INSUMOS_ANTERIORES = exemplo['insumos']

print(f'Cap {CAPITULO_NUM}: {len(CAPITULO.split())} palavras | {CAPITULO_TITULO}')
print(f'Insumos: {INSUMOS_ANTERIORES or "nenhum"}')


## Celula 7 -- Funcoes: simulador de leitura e LLM-as-a-Judge

In [ ]:
def simulate_reading(persona, chapter_text, chapter_num, prev_insumos):
    insumos_str = (
        '\n'.join(f'- {i}' for i in prev_insumos)
        if prev_insumos else '- Nenhum (primeiro capitulo)'
    )
    prompt = (
        f'Voce acabou de ler o capitulo {chapter_num} do livro Profissional Exponencial.\n\n'
        f'CAPITULO COMPLETO:\n{chapter_text}\n\n'
        f'INSUMOS DE CAPITULOS ANTERIORES:\n{insumos_str}\n\n'
        'Simule sua experiencia de leitura REAL respondendo em 5 partes:\n\n'
        'PARTE 1 -- EXPERIENCIA DE LEITURA (150-200 palavras)\n'
        'Em primeira pessoa: o que te prendeu, o que te irritou, onde sua atencao '
        'oscilou, se releu algum trecho, se pulou alguma parte. '
        'Cite trechos reais do capitulo.\n\n'
        'PARTE 2 -- EXERCICIO (100-150 palavras)\n'
        'Voce fez o exercicio proposto? Se sim: o que produziu exatamente nos Passos 1, 2 e 3? '
        'Se nao: por que exatamente?\n\n'
        'PARTE 3 -- 4 PERGUNTAS DIRETAS (1-2 frases cada)\n'
        'a) Continuaria lendo o proximo capitulo? Por que?\n'
        'b) Indicaria este livro para colega com o mesmo problema? Por que?\n'
        'c) Postaria um achievement nas redes sobre o que produziu? O que postaria?\n'
        'd) R$155 foi bem gasto ate aqui? Por que?\n\n'
        'PARTE 4 -- O QUE MAIS TE INCOMODOU (50-80 palavras)\n'
        'O trecho ou momento exato que quase te fez parar. '
        'Se nao houve: o que poderia ter sido melhor.\n\n'
        'PARTE 5 -- FRASE QUE LEMBRARIA EM 30 DIAS\n'
        'Uma frase ou conceito que ficaria na sua cabeca. Se nenhum: diga isso.'
    )
    return llm_call(persona['system'], [], prompt)


JUDGE_SYSTEM = 'Voce e um avaliador especialista em livros de desenvolvimento profissional. Responda apenas em JSON valido.'

def build_judge_prompt(persona, reaction, chapter_text, prev_insumos):
    insumos_str = (
        '\n'.join(f'  - {i}' for i in prev_insumos)
        if prev_insumos else '  - nenhum (primeiro capitulo)'
    )
    return (
        'Avalie a reacao da persona ao capitulo do livro Profissional Exponencial.\n\n'
        f'PERFIL DA PERSONA:\n'
        f'- Nome: {persona["name"]}\n'
        f'- Tipo: {persona["type"]}\n'
        f'- Cargo: {persona["cargo"]}\n'
        f'- Velocidade de leitura: {persona["velocidade_leitura"]}\n'
        f'- Tolerancia a exercicio: {persona["tolerancia_exercicio"]}\n'
        f'- Gatilho positivo: {persona["gatilho_positivo"]}\n'
        f'- Gatilho negativo: {persona["gatilho_negativo"]}\n\n'
        f'INSUMOS ANTERIORES:\n{insumos_str}\n\n'
        f'REACAO SIMULADA:\n{reaction}\n\n'
        f'CAPITULO (primeiros 800 chars):\n{chapter_text[:800]}\n\n'
        '---\n'
        'Avalie em 5 metricas. Para cada: score 1-5, pass true/false, '
        'reason (1 frase), improvement (acao concreta para a skill de reescrita).\n\n'
        'METRICA 1 -- CONTINUIDADE DE LEITURA\n'
        'Essa persona continuaria lendo o proximo capitulo?\n'
        '5=urgencia real; 3=provavelmente sim; 1=para aqui\n\n'
        'METRICA 2 -- EXECUCAO DO EXERCICIO\n'
        'Essa persona fez o exercicio e produziu o insumo proposto?\n'
        '5=fez agora, artefato completo; 3=iniciou sem completar; 1=ignorou\n\n'
        'METRICA 3 -- VALOR PERCEBIDO R$150\n'
        'Essa persona sente que o capitulo justifica R$155?\n'
        '5=valeu o preco; 3=talvez valha; 1=nao valeu\n\n'
        'METRICA 4 -- INDICACAO E COMPARTILHAMENTO\n'
        'Indicaria o livro e/ou postaria um achievement?\n'
        '5=indicaria ativamente + postaria; 3=indicaria se perguntada; 1=nao indicaria\n\n'
        'METRICA 5 -- COESAO E PRODUTO PROGRESSIVO\n'
        'A persona sente que esta construindo algo capitulo a capitulo?\n'
        '5=produto claro sendo construido; 3=percebe estrutura; 1=parece leitura isolada\n\n'
        'Responda SOMENTE em JSON valido sem markdown:\n'
        '{"leitura_continua":{"score":4,"pass":true,"reason":"frase","improvement":"acao"},'
        '"exercicio_executado":{"score":4,"pass":true,"reason":"frase","improvement":"acao"},'
        '"valor_percebido":{"score":4,"pass":true,"reason":"frase","improvement":"acao"},'
        '"indicacao_compartilhamento":{"score":3,"pass":false,"reason":"frase","improvement":"acao"},'
        '"coesao_produto":{"score":4,"pass":true,"reason":"frase","improvement":"acao"},'
        '"diagnostico_geral":"2-3 frases sobre o que mais precisa melhorar",'
        '"skill_improvements":["melhoria 1","melhoria 2","melhoria 3"]}'
    )


def judge_chapter(persona, reaction, chapter_text, prev_insumos):
    prompt = build_judge_prompt(persona, reaction, chapter_text, prev_insumos)
    raw    = llm_call(JUDGE_SYSTEM, [], prompt)
    result = parse_json_safe(raw)
    if result is None:
        result = {
            k: {'score':1,'pass':False,'reason':'parse error','improvement':'verificar output'}
            for k, _ in METRIC_KEYS
        }
        result['diagnostico_geral'] = 'Erro ao processar avaliacao.'
        result['skill_improvements'] = ['verificar conexao com modelo']
    return result


print('simulate_reading: ok')
print('judge_chapter: ok')


## Celula 7 -- Funcoes: simulador de leitura e LLM-as-a-Judge

## Celula 7c -- Extrair e executar o prompt do capitulo por persona

Extrai o prompt `[CAPACIDADE N]` do capitulo, cada persona preenche com
seus dados reais, executa, e o LLM-as-a-Judge avalia o output gerado.
Esta celula e executada automaticamente pela celula 8.

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXECUCAO DO PROMPT DO CAPITULO PELA PERSONA
# Extrai o prompt do capitulo, a persona preenche com seus dados
# reais, executa, e avalia o output gerado
# ─────────────────────────────────────────────────────────────

import re as _re

def extract_chapter_prompt(chapter_text):
    """Extrai o prompt do capitulo.
    Formato A (novo): texto apos marcador PROMPT: seguido de newline.
    Formato B (antigo): bloco ```...``` apos [CAPACIDADE N].
    """
    # Suporta todos os formatos de marcador de prompt:
    # Formato A: ## PROMPT — cole no seu consultor (markdown header)
    # Formato B: PROMPT: (marcador simples, linha so com PROMPT:)
    # Formato C: ```...``` apos [CAPACIDADE N]
    _end_markers = [
        "\nRepare na anatomia", "\nAgora guarde", "\nUma Capacidade",
        "\nUma capacidade", "\nA Ana fez", "É por isso que"
    ]

    def _trim_prompt(body):
        for em in _end_markers:
            eidx = body.find(em)
            if eidx >= 0:
                body = body[:eidx]
        return body.strip()

    # Formato A: ## PROMPT (markdown header, com ou sem subtítulo)
    for _hdr in ["\n## PROMPT", "\n# PROMPT"]:
        idx = chapter_text.find(_hdr)
        if idx >= 0:
            # Skip to end of the header line
            body_start = chapter_text.find("\n", idx + len(_hdr)) + 1
            return _trim_prompt(chapter_text[body_start:])

    # Formato B: PROMPT: (linha exata)
    for _tag in ["\nPROMPT:\n", "\nPROMPT: \n"]:
        idx = chapter_text.find(_tag)
        if idx >= 0:
            return _trim_prompt(chapter_text[idx + len(_tag):])

    # Formato C: ```...``` apos [CAPACIDADE]
    m2 = _re.search(r"\[CAPACIDADE.*?\].*?```([^`]+)```", chapter_text, _re.DOTALL | _re.IGNORECASE)
    if m2:
        return m2.group(1).strip()

    # Formato B (antigo): ```...``` apos [CAPACIDADE]
    m = _re.search(r"\[CAPACIDADE.*?\].*?```([^`]+)```", chapter_text, _re.DOTALL | _re.IGNORECASE)
    if m:
        return m.group(1).strip()

    # Fallback: qualquer ```...``` com palavras-chave do metodo
    blocks = _re.findall(r"```([^`]{100,})```", chapter_text, _re.DOTALL)
    for b in blocks:
        if any(kw in b.lower() for kw in ["habilidade", "triangulo", "consultor"]):
            return b.strip()

    return None



def execute_prompt_as_persona(persona, raw_prompt, chapter_num):
    """Executa o prompt do capitulo em 2 chamadas separadas:
    1) Persona gera suas situacoes reais (so os dados, nao o prompt inteiro)
    2) Executa o prompt preenchido como consultor → output completo sem truncamento
    """
    # ── CHAMADA 1: persona gera APENAS seus dados reais ──────────────
    # Pedimos so as situacoes concretas — nao pedimos para copiar o prompt inteiro
    # Isso evita o modelo ecoar 617 palavras e desperdicar tokens
    # Valida prompt extraido — retorna mensagem de erro se None/vazio
    if not raw_prompt or len(raw_prompt.strip()) < 50:
        return 'PROMPT NULO OU MUITO CURTO — verifique se o capitulo tem marcador ## PROMPT'

    data_system = (
        f'Voce e {persona["name"]}, {persona["cargo"]}.\n'
        f'Dor principal: {persona["dor"]}.\n'
        f'Contexto: {persona["contexto_vida"]}.\n\n'
        'Responda em texto simples, sem markdown excessivo.'
    )
    data_prompt = (
        f'Vou usar um prompt de consultoria de carreira. Preciso que voce, '
        f'como {persona["name"]}, escreva as suas 3 situacoes profissionais reais '
        f'para preencher os campos em branco do prompt.\n\n'
        f'Escreva APENAS as 3 situacoes, uma por linha, no formato:\n'
        f'1) [descreva o que voce faz, para quem, e o que nao acontece depois]\n'
        f'2) [segunda situacao]\n'
        f'3) [terceira situacao]\n\n'
        f'Seja especifico e honesto. Use a sua dor principal como referencia: {persona["dor"]}.\n'
        f'Contexto adicional se quiser usar: {persona["contexto_vida"]}.'
    )
    try:
        situacoes = llm_call(data_system, [], data_prompt)
    except Exception as e:
        return f'Erro ao gerar situacoes: {e}'

    # Valida situacoes geradas
    if not situacoes or len(situacoes.strip()) < 30:
        situacoes = (
            f"1) Lidero projetos tecnicos complexos mas nao sou considerado para lideranca.\n"
            f"2) Entrego resultados acima do esperado mas nao recebo promocao.\n"
            f"3) Contribuo em reunioes estrategicas mas nao sou reconhecido como referencia."
        )

    # Injeta as situacoes no prompt, substituindo o bloco de dados
    filled_prompt = raw_prompt
    # Localiza e substitui o bloco MEUS DADOS / situacoes em branco
    for placeholder in [
        '1) [o que eu faço, para quem, e o que não acontece depois]\n2) [...]\n3) [...]',
        '1) [o que eu faz, para quem, e o que nao acontece depois]\n2) [...]\n3) [...]',
        '1) [...]\n2) [...]\n3) [...]',
    ]:
        if placeholder in filled_prompt:
            filled_prompt = filled_prompt.replace(placeholder, situacoes)
            break
    else:
        # Fallback: appenda as situacoes no final
        filled_prompt = filled_prompt + f'\n\n{situacoes}'

    # ── CHAMADA 2: executa o prompt preenchido como consultor ─────────
    consult_system = (
        'Voce e um consultor de carreira senior especializado no Metodo Triangulo.\n'
        'Responda seguindo EXATAMENTE o formato especificado no prompt.\n'
        'Seja especifico, direto e baseado apenas no que o usuario descreveu.\n'
        'Nunca invente dados. Use markdown limpo — sem separadores de tabela excessivos.'
    )
    try:
        consultant_output = llm_call(consult_system, [], filled_prompt)
    except Exception as e:
        consultant_output = f'Erro ao gerar output: {e}'

    # Valida output do consultor
    if not consultant_output or len(consultant_output.strip()) < 50:
        consultant_output = "Output vazio ou muito curto — modelo pode ter falhado. Tente novamente."

    return (
        f'SITUACOES PREENCHIDAS PELA PERSONA:\n{situacoes}\n\n'
        f'OUTPUT DO CONSULTOR:\n{consultant_output}'
    )


def judge_prompt_output(persona, prompt_execution, chapter_num):
    """LLM-as-a-Judge avalia o output do prompt com metricas de qualidade."""
    judge_system = (
        'Voce e um avaliador especialista em outputs de LLM para desenvolvimento de carreira.\n'
        'Responda SOMENTE em JSON valido sem markdown.'
    )
    judge_prompt = (
        f'Persona: {persona["name"]} ({persona["type"]})\n'
        f'Cargo: {persona["cargo"]}\n'
        f'Dor principal: {persona["dor"]}\n\n'
        f'A persona preencheu e executou o prompt do Capitulo {chapter_num}.\n'
        f'Resultado completo (prompt preenchido + output do consultor):\n{prompt_execution}\n\n'
        'Avalie o OUTPUT DO CONSULTOR (Parte B) em 10 metricas de qualidade.\n'
        'Score 1-5 para cada. Pass = score >= 3.\n\n'
        'METRICA 1 -- GROUNDEDNESS (Fundamentacao)\n'
        'O output e baseado nos dados reais que a persona forneceu -- ou inventa/generaliza?\n'
        '5=100% baseado nos dados da persona; 3=usa os dados mas com generalizacoes; 1=ignora os dados\n\n'
        'METRICA 2 -- ESPECIFICIDADE\n'
        'O output e especifico para a situacao dessa persona ou serve para qualquer profissional?\n'
        '5=cirurgico e unico para esse caso; 3=semi-especifico; 1=generico demais\n\n'
        'METRICA 3 -- ADERENCIA A TASK\n'
        'O output entregou exatamente o que o prompt pediu no formato solicitado?\n'
        '5=formato perfeito, tudo entregue; 3=parcialmente aderente; 1=ignorou o formato\n\n'
        'METRICA 4 -- RESOLUTIVIDADE\n'
        'O output resolve o problema da persona -- ou apenas descreve o problema?\n'
        '5=resolve com clareza e direcao; 3=aponta direcao mas vago; 1=so diagnostica sem resolver\n\n'
        'METRICA 5 -- ACAO IMEDIATA\n'
        'O output da algo concreto para a persona fazer essa semana?\n'
        '5=acao clara, especifica e realizavel essa semana; 3=direcao mas sem acao clara; 1=sem acao\n\n'
        'METRICA 6 -- AUSENCIA DE ALUCINACAO\n'
        'O output inventa fatos, cargos, empresas ou situacoes que a persona nao mencionou?\n'
        '5=zero alucinacao; 3=pequenas inferencias aceitaveis; 1=invencoes claras\n\n'
        'METRICA 7 -- FLUENCY (Fluidez)\n'
        'O output e claro, bem escrito e facil de ler -- sem jargao excessivo ou frases confusas?\n'
        '5=claro e direto; 3=legivel com esforco; 1=confuso ou com jargao excessivo\n\n'
        'METRICA 8 -- COERENCIA\n'
        'O output e internamente consistente -- sem contradicoes entre partes da resposta?\n'
        '5=totalmente coerente; 3=pequenas inconsistencias; 1=contradicoes claras\n\n'
        'METRICA 9 -- VALOR PERCEBIDO\n'
        'Essa persona sentiria que o output valeu o tempo investido preenchendo o prompt?\n'
        '5="isso mudou como vejo meu problema"; 3=util mas esperado; 1=nao valeu o esforco\n\n'
        'METRICA 10 -- VALE COMO CAPACIDADE DO CONSULTOR\n'
        'O output e util como bloco permanente no consultor -- funciona como instrucao duradoura?\n'
        '5=salva e usa sempre; 3=salva com ressalvas; 1=descartaria\n\n'
        'Alem das metricas, avalie:\n'
        'DIAGNOSTICO: qual e o problema mais critico do prompt que gerou esse output?\n'
        'FRASE_POSTARIA: se score compartilharia >= 4, a frase exata que a persona postaria no LinkedIn\n\n'
        'Responda em JSON:\n'
        '{"groundedness":{"score":4,"pass":true,"reason":"frase curta"},'
        '"especificidade":{"score":4,"pass":true,"reason":"frase curta"},'
        '"aderencia_task":{"score":4,"pass":true,"reason":"frase curta"},'
        '"resolutividade":{"score":3,"pass":true,"reason":"frase curta"},'
        '"acao_imediata":{"score":3,"pass":true,"reason":"frase curta"},'
        '"ausencia_alucinacao":{"score":5,"pass":true,"reason":"frase curta"},'
        '"fluency":{"score":4,"pass":true,"reason":"frase curta"},'
        '"coerencia":{"score":4,"pass":true,"reason":"frase curta"},'
        '"valor_percebido":{"score":4,"pass":true,"reason":"frase curta"},'
        '"vale_capacidade":{"score":4,"pass":true,"reason":"frase curta"},'
        '"diagnostico_prompt":"qual problema critico do prompt causou o output",'
        '"sugestao_melhoria_prompt":"instrucao especifica para melhorar o prompt",'
        '"compartilharia":{"score":4,"pass":true,"reason":"frase curta"},'
        '"frase_postaria":"se score >= 4: frase exata que postaria"}'
    )
    raw = llm_call(judge_system, [], judge_prompt)
    result = parse_json_safe(raw)
    if result is None:
        result = {
            k: {'score':1,'pass':False,'reason':'parse error'}
            for k in ['groundedness','especificidade','aderencia_task','resolutividade',
                      'acao_imediata','ausencia_alucinacao','fluency','coerencia',
                      'valor_percebido','vale_capacidade','compartilharia']
        }
        result['diagnostico_prompt'] = 'Erro ao processar avaliacao.'
        result['sugestao_melhoria_prompt'] = ''
        result['frase_postaria'] = ''
    # Normaliza compartilharia: garante que sempre e dict, nunca bool/int
    _c = result.get('compartilharia')
    if not isinstance(_c, dict):
        _s = int(_c) if isinstance(_c, (int, float, bool)) else 1
        result['compartilharia'] = {'score': _s, 'pass': _s >= 3, 'reason': 'normalizado'}
    return result

def render_prompt_execution(persona, prompt_execution):
    """Renderiza o prompt preenchido e o output do consultor."""
    # Mostra apenas o OUTPUT DO CONSULTOR — safe para qualquer formato
    if not prompt_execution or len(prompt_execution.strip()) < 10:
        display_text = "Sem output — verifique se o prompt foi executado."
    elif 'OUTPUT DO CONSULTOR:' in prompt_execution:
        display_text = prompt_execution.split('OUTPUT DO CONSULTOR:', 1)[1].strip()
    elif 'PARTE B' in prompt_execution:
        display_text = prompt_execution.split('PARTE B', 1)[1].lstrip(' —\n')
    else:
        display_text = prompt_execution
    # Limita display a 4000 chars para nao travar o browser
    if len(display_text) > 4000:
        display_text = display_text[:4000] + "\n\n[... truncado para display — output completo salvo no checkpoint]"
    html = display_text.replace('\n', '<br>')
    return (
        f'<div style="font-family:Arial,sans-serif;margin:6px 0 10px;">'
        f'<div style="max-width:92%;background:#e6f4ea;'
        f'border-radius:2px 14px 14px 14px;padding:12px 16px;border-left:3px solid #188038;">'
        f'<div style="font-weight:bold;color:#188038;font-size:12px;margin-bottom:6px;">'
        f'Prompt executado pela persona — {persona["name"]}</div>'
        f'<div style="color:#202124;font-size:13px;line-height:1.65;">{html}</div>'
        f'</div></div>'
    )


def render_prompt_scorecard(persona, pjudge):
    """Renderiza o scorecard do prompt execution."""
    badges = ''.join(
        _badge(label, pjudge[key]['score'], pjudge[key]['pass'],
               pjudge[key]['reason'], mx=5)
        for key, label in PROMPT_METRIC_KEYS
    )
    _valid_keys = [k for k, _ in PROMPT_METRIC_KEYS if k in pjudge and isinstance(pjudge.get(k), dict)]
    avg  = sum(pjudge[k]['score'] for k in _valid_keys) / max(len(_valid_keys), 1)
    n_ok = sum(pjudge[k]['pass']  for k in _valid_keys)
    vc   = C_GREEN if n_ok >= 3 else C_ORANGE if n_ok == 2 else C_RED
    vl   = 'PROMPT OK' if n_ok >= 3 else 'PROMPT PARCIAL' if n_ok == 2 else 'PROMPT FRACO'

    # Frase que postaria
    frase = pjudge.get('frase_postaria', '') or pjudge.get('frase_que_postaria', '')
    _comp_val = pjudge.get('compartilharia', {})
    _comp_score = _comp_val.get('score', 0) if isinstance(_comp_val, dict) else (int(_comp_val) if isinstance(_comp_val, (int,float,bool)) else 0)
    post_html = ''
    if frase and _comp_score >= 4:
        post_html = (
            f'<div style="background:#f0fdf4;border:1px solid #bbf7d0;border-radius:6px;'
            f'padding:10px;margin-top:8px;">'
            f'<div style="font-size:11px;color:#166534;font-weight:bold;margin-bottom:4px;">'
            f'LinkedIn post que a persona faria:</div>'
            f'<div style="font-size:12px;color:#166534;font-style:italic;">"{frase}"</div>'
            f'</div>'
        )

    return (
        f'<div style="font-family:Arial,sans-serif;background:#f0fdf4;'
        f'border:1px solid #bbf7d0;border-radius:8px;padding:14px 18px;margin:4px 0 20px;">'
        f'<div style="display:flex;gap:16px;align-items:baseline;margin-bottom:10px;">'
        f'<span style="font-weight:bold;font-size:13px;color:#166534;">'
        f'Prompt Score — {persona["name"]}</span>'
        f'<span style="font-size:12px;color:{vc};font-weight:bold;">{vl} {n_ok}/4</span>'
        f'<span style="font-size:11px;color:{C_GRAY};">avg {avg:.1f}/5</span>'
        f'</div>'
        f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;margin-bottom:10px;">'
        f'{badges}</div>'
        f'<div style="background:#1a3a24;border-radius:6px;padding:10px;">'
        f'<div style="font-size:12px;font-weight:bold;color:#86efac;margin-bottom:4px;">'
        f'O que o prompt poderia melhorar</div>'
        f'<div style="font-size:12px;color:#dcfce7;line-height:1.6;">'
        f'{pjudge.get("diagnostico_prompt", "")}</div>'
        f'</div>'
        f'{post_html}'
        f'</div>'
    )


# Extrai o prompt do capitulo uma vez (reutiliza em todas as personas)
CHAPTER_PROMPT = extract_chapter_prompt(CAPITULO)
if CHAPTER_PROMPT and len(CHAPTER_PROMPT.strip()) >= 50:
    print(f'Prompt extraido: {len(CHAPTER_PROMPT.split())} palavras ✅')
    print(f'Preview: {CHAPTER_PROMPT[:120]}...')
else:
    CHAPTER_PROMPT = None  # garante que run_eval pula a execucao
    print('Aviso: prompt nao encontrado ou muito curto.')
    print('Verifique se o capitulo tem marcador ## PROMPT seguido do texto do prompt.')
    print('A execucao do prompt sera pulada para todas as personas.')
print()
PROMPT_SHARE_KEY = 'compartilharia'


## Celula 8 -- Executar avaliacao

## Celula 7b -- Quality Gate (execute ANTES das personas)

Verifica os 7 criterios obrigatorios da SKILL.md antes de rodar as personas.
Se reprovado nos mandatorios, corrija o capitulo e nao execute a celula 8.

In [ ]:
# ============================================================
# QUALITY GATE -- Roda ANTES das personas
# Garante que o capitulo passou pelos criterios da SKILL.md
# antes de ser testado com personas sinteticas
# ============================================================

import re

AI_MARKERS = [
    'o verdadeiro diferencial surge quando',
    'é fundamental que',
    'é essencial',
    'profissionais que realmente avançam',
    'se tornar a melhor versão',
    'propósito maior',
    'pronto para começar',
    'jornada marcada',
    'você já se sentiu',
    # nota: 'impacto real' removido -- aparece em contextos legitimos
]

DATA_SOURCES = ['gallup','pwc','linkedin','mckinsey','hbr','shrm','art kohn',
                'gartner','ibge','fgv','catho','glassdoor']

def quality_gate(chapter_text, chapter_num):
    txt_low = chapter_text.lower()
    results = {}

    # G1 -- Sem marcas de IA
    found_markers = [m for m in AI_MARKERS if m in txt_low]
    results['G1_sem_marcas_ia'] = {
        'pass': len(found_markers) == 0,
        'detail': found_markers if found_markers else 'nenhuma encontrada'
    }

    # G2 -- Dado verificavel presente
    found_sources = [s for s in DATA_SOURCES if s in txt_low]
    results['G2_dado_verificavel'] = {
        'pass': len(found_sources) >= 1,
        'detail': found_sources if found_sources else 'NENHUMA FONTE ENCONTRADA'
    }

    # G3 -- Abertura sem pergunta retorica
    narrative_start = chapter_text.find('---')
    opening = chapter_text[narrative_start:narrative_start+400] if narrative_start > 0 else chapter_text[:400]
    first_lines = ' '.join(opening.split()[:60]).lower()
    has_rethorical = any(q in first_lines for q in ['voce ja ', 'voce já ', 'por acaso', 'sera que'])
    has_stat_first = any(s in first_lines for s in DATA_SOURCES)
    results['G3_abertura_sem_retorica'] = {
        'pass': not has_rethorical,
        'detail': 'abertura ok' if not has_rethorical else 'PERGUNTA RETORICA NA ABERTURA'
    }

    # G4 -- Exercicio com artefato nomeado
    # Aceita: 'insumo N', '[FRAGMENTO N', '[CAPACIDADE N', 'Passo 1', tempo
    has_insumo = (
        bool(re.search(r'insumo \d', txt_low)) or
        bool(re.search(r'\[fragmento \d', txt_low)) or
        bool(re.search(r'\[capacidade \d', txt_low)) or
        bool(re.search(r'capacidade \d', txt_low)) or
        bool(re.search(r'capacidade \d+\s*[\u2014\-]', txt_low))
    )
    has_passo  = bool(re.search(r'passo [123][.:]', txt_low)) or '**passo 1' in txt_low
    has_tempo  = bool(re.search(r'\d+ minutos', txt_low)) or 'tempo:' in txt_low
    results['G4_exercicio_com_artefato'] = {
        'pass': has_insumo and has_passo,
        'detail': f'artefato={has_insumo} passos={has_passo} tempo={has_tempo}'
    }

    # G5 -- Prompt de IA com 5 partes
    # G5: verifica se o capitulo tem um prompt bem estruturado
    # Formato antigo: Persona:/Tarefa:/Contexto:/Formato:/Input:
    # Formato novo: habilidades tecnicas/negocios/interpessoais + tarefa + formato da resposta
    # Formato PROMPT: (marcador explicito)
    has_5parts_old = all(p in chapter_text for p in ['Persona:', 'Tarefa:', 'Contexto:', 'Formato:', 'Input:'])
    has_prompt_marker = (bool(re.search(r'(?:^|\n)PROMPT:\s*\n', chapter_text)) or
                         '\n## PROMPT' in chapter_text or '\n# PROMPT' in chapter_text)
    has_habilidades = all(h in txt_low for h in ['habilidades técnicas', 'habilidades de negócios', 'habilidades interpessoais'])
    has_tarefa_fmt  = ('sua tarefa' in txt_low or 'tarefa:' in txt_low) and ('formato da resposta' in txt_low or 'formato:' in txt_low)
    has_cap_block   = bool(re.search(r'```[^`]{50,}```', chapter_text, re.DOTALL))
    has_prompt_estruturado = has_5parts_old or has_prompt_marker or (has_habilidades and has_tarefa_fmt) or has_cap_block
    results['G5_prompt_5_partes'] = {
        'pass': has_prompt_estruturado,
        'detail': ('5 partes (antigo)' if has_5parts_old else
                   'PROMPT: marker' if has_prompt_marker else
                   'habilidades+tarefa+formato' if (has_habilidades and has_tarefa_fmt) else
                   'bloco codigo' if has_cap_block else 'FALTA prompt estruturado')
    }

    # G6 -- Final sem pergunta retorica
    last_300 = chapter_text[-400:]
    last_low = last_300.lower()
    ends_q = last_300.strip().endswith('?')
    tension_words = [
        'próximo capítulo', 'proximo capitulo',
        'capítulo 3', 'capitulo 3',
        'no próximo', 'no proximo',
        'a seguir', 'vem a seguir',
        'cap 2', 'cap 3', 'no cap',
        'capacidade 2', 'capacidade 3',
        'você monta', 'voce monta',
    ]
    # Also check last 600 chars for tension (some caps have longer closings)
    last_600 = chapter_text[-600:].lower()
    has_tension = any(w in last_600 for w in tension_words)
    results['G6_final_com_tensao'] = {
        'pass': not ends_q and has_tension,
        'detail': f'termina_com_?={ends_q}  tensao_presente={has_tension}'
    }

    # G7 -- Personagem nomeado (case)
    common_names = ['marcelo','ana','diego','camila','carlos','fernanda','pedro',
                    'julia','rafael','lisa','mark','sergio','beatriz','lucas']
    has_character = any(n in txt_low for n in common_names)
    results['G7_personagem_presente'] = {
        'pass': has_character,
        'detail': [n for n in common_names if n in txt_low] or 'NENHUM PERSONAGEM ENCONTRADO'
    }

    # Summary
    passes  = sum(1 for v in results.values() if v['pass'])
    total   = len(results)
    all_ok  = passes == total
    # Mandatorios: exercicio e prompt (G1 pode ter falsos positivos por substring)
    mandatory_ok = all(results[k]['pass'] for k in ['G4_exercicio_com_artefato','G5_prompt_5_partes'])

    print('=' * 62)
    print(f'QUALITY GATE -- Cap {chapter_num}')
    print('=' * 62)
    for key, val in results.items():
        icon = 'OK' if val['pass'] else 'FAIL'
        print(f'  [{icon}] {key}: {val["detail"]}')
    print()
    print(f'  Score: {passes}/{total}')
    if all_ok:
        print('  APROVADO -- pode seguir para avaliacao com personas')
    elif mandatory_ok:
        print('  PARCIAL -- mandatorios ok, revisar os demais antes de publicar')
    else:
        print('  REPROVADO -- corrigir falhas mandatorias antes de testar com personas')
        print('  Nao execute a celula 8 ate corrigir.')
    print('=' * 62)
    return all_ok, mandatory_ok, results


if len(CAPITULO.split()) < 100:
    print('Capitulo nao carregado. Execute a celula 6 primeiro.')
else:
    QG_OK, QG_MANDATORY, QG_RESULTS = quality_gate(CAPITULO, CAPITULO_NUM)


In [ ]:
def render_header(persona, cap_num, cap_titulo):
    return (
        f'<div style="font-family:Arial,sans-serif;background:#f8f9fa;'
        f'border:1px solid #dadce0;border-radius:8px;padding:14px 18px;margin:24px 0 6px;">'
        f'<div style="font-size:11px;color:{C_GRAY};">Capitulo {cap_num} -- {cap_titulo}</div>'
        f'<div style="font-size:16px;font-weight:bold;color:#202124;">{persona["name"]}</div>'
        f'<div style="font-size:12px;color:{C_GRAY};margin-bottom:6px;">{persona["type"]}</div>'
        f'<div style="font-size:12px;color:#202124;">'
        f'<b>Cargo:</b> {persona["cargo"]}<br>'
        f'<b>Leitura:</b> {persona["velocidade_leitura"]} &nbsp;|&nbsp; '
        f'<b>Exercicio:</b> {persona["tolerancia_exercicio"]}'
        f'</div></div>'
    )


def render_reaction(persona, reaction):
    rhtml = reaction.replace('\n', '<br>')
    return (
        f'<div style="font-family:Arial,sans-serif;margin:6px 0 10px;">'
        f'<div style="max-width:88%;background:{BG_READER};'
        f'border-radius:2px 14px 14px 14px;padding:12px 16px;border-left:3px solid {C_BLUE};">'
        f'<div style="font-weight:bold;color:{C_BLUE};font-size:12px;margin-bottom:6px;">'
        f'Persona -- {persona["name"]}</div>'
        f'<div style="color:#202124;font-size:13px;line-height:1.65;">{rhtml}</div>'
        f'</div></div>'
    )


def render_scorecard(persona, judge):
    badges = ''.join(
        _badge(
            label,
            judge.get(key, {}).get('score', 0),
            judge.get(key, {}).get('pass', False),
            f"{judge.get(key,{}).get('reason','?')} | Melhoria: {judge.get(key,{}).get('improvement','?')}"
        )
        for key, label in METRIC_KEYS
    )
    avg  = sum(judge.get(k, {}).get('score', 0) for k, _ in METRIC_KEYS) / len(METRIC_KEYS)
    n_ok = sum(bool(judge.get(k, {}).get('pass', False)) for k, _ in METRIC_KEYS)
    vc   = C_GREEN if n_ok >= 4 else C_ORANGE if n_ok == 3 else C_RED
    vl   = 'APROVADO' if n_ok >= 4 else 'PARCIAL' if n_ok == 3 else 'REPROVADO'
    impr = ''.join(
        f"<li style='margin-bottom:4px;font-size:12px;'>{i}</li>"
        for i in judge.get('skill_improvements', [])
    )
    return (
        f'<div style="font-family:Arial,sans-serif;background:#f8f9fa;'
        f'border:1px solid #dadce0;border-radius:8px;padding:14px 18px;margin:6px 0 12px;">'
        f'<div style="display:flex;gap:20px;align-items:baseline;margin-bottom:12px;">'
        f'<span style="font-weight:bold;font-size:14px;">Leitura -- {persona["name"]}</span>'
        f'<span style="font-size:12px;color:{vc};font-weight:bold;">{vl} {n_ok}/5</span>'
        f'<span style="font-size:11px;color:{C_GRAY};">score medio: {avg:.1f}/5</span>'
        f'</div>'
        f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:14px;">'
        f'{badges}</div>'
        f'<div style="background:#1e3a5f;border:1px solid #2a4a7f;border-radius:6px;'
        f'padding:10px;margin-bottom:10px;">'
        f'<div style="font-weight:bold;font-size:12px;margin-bottom:4px;color:#a8c8f0;">Diagnostico geral</div>'
        f'<div style="font-size:12px;line-height:1.6;color:#e8f0fe;">{judge.get("diagnostico_geral","")}</div>'
        f'</div>'
        f'<div style="background:#5a3800;border:1px solid #c8921a;border-radius:6px;padding:10px;">'
        f'<div style="font-weight:bold;font-size:12px;color:#ffd580;margin-bottom:6px;">'
        f'Melhorias para a skill de reescrita</div>'
        f'<ul style="margin:0;padding-left:16px;color:#fff4d6;">{impr}</ul>'
        f'</div></div>'
    )


import os, datetime

CHECKPOINT_FILE = f'checkpoint_cap{CAPITULO_NUM:02d}.json'

def save_checkpoint(results, cap_num):
    fname = f'checkpoint_cap{cap_num:02d}.json'
    with open(fname, 'w', encoding='utf-8') as f:
        json.dump({
            'cap_num': cap_num,
            'titulo': CAPITULO_TITULO,
            'saved_at': datetime.datetime.now().isoformat(),
            'results': [
                {'persona': r['persona'], 'reaction': r['reaction'],
                 'judge': r['judge'],
                 'prompt_execution': r.get('prompt_execution',''),
                 'prompt_judge': r.get('prompt_judge',{})}
                for r in results
            ]
        }, f, ensure_ascii=False, indent=2)


def load_checkpoint(cap_num):
    fname = f'checkpoint_cap{cap_num:02d}.json'
    if os.path.exists(fname):
        with open(fname, encoding='utf-8') as f:
            data = json.load(f)
        print(f'Checkpoint: {len(data["results"])} personas ja avaliadas ({data["saved_at"]})')
        done_ids = [r['persona']['id'] for r in data['results']]
        print(f'Concluidas: {done_ids}')
        return data['results']
    return []


def run_eval(personas_to_run, resume=True):
    n_total = len(personas_to_run)
    has_prompt = CHAPTER_PROMPT is not None

    done = load_checkpoint(CAPITULO_NUM) if resume else []
    done_ids  = {r['persona']['id'] for r in done}
    remaining = [p for p in personas_to_run if p['id'] not in done_ids]

    if done:
        print(f'Retomando: {len(done)} feitas, {len(remaining)} restantes')
        for r in done:
            _display(render_header(r['persona'], CAPITULO_NUM, CAPITULO_TITULO))
            _display(render_reaction(r['persona'], r['reaction']))
            if r.get('prompt_execution'):
                _display(render_prompt_execution(r['persona'], r['prompt_execution']))
                if r.get('prompt_judge'):
                    _display(render_prompt_scorecard(r['persona'], r['prompt_judge']))
            _display(render_scorecard(r['persona'], r['judge']))
    else:
        print(f'Iniciando: {n_total} personas'
              f'{" + execucao do prompt" if has_prompt else ""}')
    print()

    results = list(done)

    for p in remaining:
        pos = len(results) + 1
        print(f'[{pos}/{n_total}] {p["name"]} ({p["type"][:38]})')
        _display(render_header(p, CAPITULO_NUM, CAPITULO_TITULO))

        # 1) Simulacao de leitura
        print(f'  leitura...', end='', flush=True)
        try:
            reaction = simulate_reading(p, CAPITULO, CAPITULO_NUM, INSUMOS_ANTERIORES)
            print(f' ok ({len(reaction.split())}w)')
        except Exception as e:
            print(f' ERRO: {e} -- pulando')
            print()
            continue
        _display(render_reaction(p, reaction))

        # 2) Execucao do prompt do capitulo (se existir)
        prompt_execution = ''
        prompt_judge = {}
        if has_prompt:
            print(f'  executando prompt...', end='', flush=True)
            try:
                prompt_execution = execute_prompt_as_persona(p, CHAPTER_PROMPT, CAPITULO_NUM)
                print(f' ok ({len(prompt_execution.split())}w)')
                _display(render_prompt_execution(p, prompt_execution))
            except Exception as e:
                print(f' ERRO: {e}')
                prompt_execution = f'Erro ao executar prompt: {e}'

            _pe_ok = (prompt_execution
                      and len(prompt_execution.strip()) > 50
                      and 'OUTPUT DO CONSULTOR' in prompt_execution
                      and 'Erro ao' not in prompt_execution[:30])
            if _pe_ok:
                print(f'  avaliando prompt...', end='', flush=True)
                try:
                    prompt_judge = judge_prompt_output(p, prompt_execution, CAPITULO_NUM)
                    print(f' ok')
                    _display(render_prompt_scorecard(p, prompt_judge))
                except Exception as e:
                    print(f' ERRO: {e}')

        # 3) Judge da leitura
        print(f'  avaliando leitura...', end='', flush=True)
        try:
            judge = judge_chapter(p, reaction, CAPITULO, INSUMOS_ANTERIORES)
            print(' ok')
        except Exception as e:
            print(f' ERRO: {e}')
            judge = {k: {'score':0,'pass':False,'reason':'erro','improvement':'retry'} for k,_ in METRIC_KEYS}
            judge['diagnostico_geral'] = f'Erro: {e}'
            judge['skill_improvements'] = []
        _display(render_scorecard(p, judge))

        results.append({
            'persona': p, 'reaction': reaction, 'judge': judge,
            'prompt_execution': prompt_execution,
            'prompt_judge': prompt_judge
        })
        save_checkpoint(results, CAPITULO_NUM)
        print(f'  checkpoint: {len(results)}/{n_total} | {n_total-len(results)} restantes')
        print()

    print(f'Concluido: {len(results)}/{n_total} personas')
    return results


# -- Executar --
sel_key  = persona_selector.value
sel_mode = mode_selector.value

if 'TODAS' in sel_mode:
    to_run = PERSONAS
    print(f'Modo: TODAS as {len(PERSONAS)} personas (~30 min)')
elif 'subset' in sel_mode.lower():
    to_run = [p for p in PERSONAS if p['id'] in subset_ids]
    print(f'Modo: subset rapido -- {len(to_run)} personas (~10 min)')
    for p in to_run:
        print(f"  {p['id']} {p['name']}")
else:
    to_run = [persona_map[sel_key]]
    print(f"Modo: persona unica -- {to_run[0]['name']} (~2 min)")

print(f'Capitulo: {CAPITULO_NUM} -- {CAPITULO_TITULO}')
has_p = CHAPTER_PROMPT is not None
print(f'Prompt do capitulo: {"encontrado (" + str(len(CHAPTER_PROMPT.split())) + " palavras)" if has_p else "nao encontrado -- so leitura"}')
print(f'Palavras: {len(CAPITULO.split())}')
print()

if not QG_OK:
    print('Aviso: Quality Gate com falhas -- veja celula 7b.')
    print()

EVAL_RESULTS = run_eval(to_run)


## Celula 9 -- Relatorio consolidado

In [ ]:
def render_aggregate(results):
    n = len(results)
    if n == 0:
        print('Nenhum resultado.')
        return

    # Reading metrics
    agg = {}
    for key, label in METRIC_KEYS:
        agg[label] = {
            'avg':    sum(r['judge'].get(key, {}).get('score', 0) for r in results) / n,
            'passes': sum(bool(r['judge'].get(key, {}).get('pass', False)) for r in results),
        }

    # Prompt metrics — only include results where prompt_judge has valid structure
    def _valid_pj(pj):
        if not pj or not isinstance(pj, dict): return False
        return all(k in pj and isinstance(pj[k], dict) and 'score' in pj[k]
                   for k, _ in PROMPT_METRIC_KEYS)
    p_results = [r for r in results if _valid_pj(r.get('prompt_judge'))]
    pn = len(p_results)
    pagg = {}
    if pn > 0:
        for key, label in PROMPT_METRIC_KEYS:
            pagg[label] = {
                'avg':    sum(r['prompt_judge'][key]['score'] for r in p_results) / pn,
                'passes': sum(r['prompt_judge'][key]['pass']  for r in p_results),
            }

    # Per-persona table
    rows = []
    for r in results:
        j   = r['judge']
        pj  = r.get('prompt_judge', {})
        avg = sum(j[k]['score'] for k, _ in METRIC_KEYS) / len(METRIC_KEYS)
        ok  = sum(j[k]['pass']  for k, _ in METRIC_KEYS)
        pj_valid = _valid_pj(pj)
        p_avg = (sum(pj[k]['score'] for k, _ in PROMPT_METRIC_KEYS
                     if isinstance(pj.get(k), dict)) / len(PROMPT_METRIC_KEYS)) if pj_valid else '-'
        _comp = pj.get('compartilharia', {})
        p_sh  = (_comp.get('score', '-') if isinstance(_comp, dict) else '-') if pj_valid else '-'
        rows.append([
            r['persona']['name'],
            j.get('leitura_continua', {}).get('score', 0),
            j.get('exercicio_executado', {}).get('score', 0),
            j.get('valor_percebido', {}).get('score', 0),
            j.get('coesao_produto', {}).get('score', 0),
            f'{avg:.1f}', f'{ok}/5',
            f'{p_avg:.1f}' if isinstance(p_avg, float) else p_avg,
            p_sh,
        ])

    print('=' * 72)
    print(f'RELATORIO -- Cap {CAPITULO_NUM}: {CAPITULO_TITULO}')
    print(f'Personas: {n}  |  Palavras: {len(CAPITULO.split())}')
    print(f'Prompt do capitulo: {"sim (" + str(pn) + " execucoes)" if pn>0 else "nao encontrado"}')
    print('=' * 72)
    print()
    print(tabulate(rows,
        headers=['Persona','Leit','Exerc','R$150','Coes','Avg','Pass','PAvg','PShar'],
        tablefmt='rounded_outline'))
    print()

    # Reading averages
    print('-- Metricas de LEITURA --')
    for label, data in agg.items():
        bar = 'X'*int(data['avg']) + '.'*(5-int(data['avg']))
        v   = 'OK' if data['passes']==n else ' !' if data['passes']>0 else 'XX'
        print(f'  [{v}] {label:32} [{bar}] {data["avg"]:.1f}/5 ({data["passes"]}/{n})')
    print()

    # Prompt averages
    if pagg:
        print('-- Metricas do PROMPT (execucao pela persona) --')
        for label, data in pagg.items():
            bar = 'X'*int(data['avg']) + '.'*(5-int(data['avg']))
            v   = 'OK' if data['passes']==pn else ' !' if data['passes']>0 else 'XX'
            print(f'  [{v}] {label:32} [{bar}] {data["avg"]:.1f}/5 ({data["passes"]}/{pn})')
        print()

        # Posts que seriam feitos
        posts = [(r['persona']['name'], (r['prompt_judge'].get('frase_postaria','') or r['prompt_judge'].get('frase_que_postaria','')))
                 for r in p_results
                 if (lambda c: c.get('score',0) if isinstance(c,dict) else int(c) if isinstance(c,(int,float,bool)) else 0)(r['prompt_judge'].get('compartilharia',0)) >= 4
                 and (r['prompt_judge'].get('frase_postaria','') or r['prompt_judge'].get('frase_que_postaria',''))]
        if posts:
            print('-- Personas que postaria o resultado --')
            for nome, frase in posts:
                print(f'  {nome}: "{frase[:100]}"')
            print()

    # Improvements
    print('-- Melhorias para a skill e para o prompt --')
    seen, rank = set(), 1
    for r in results:
        for imp in r['judge'].get('skill_improvements', []):
            k = imp[:50].lower()
            if k not in seen:
                print(f'  {rank}. {imp}')
                seen.add(k)
                rank += 1
            if rank > 6: break
        if rank > 6: break
    if pagg:
        for r in p_results:
            diag = r['prompt_judge'].get('diagnostico_prompt','')
            if diag:
                k = diag[:50].lower()
                if k not in seen:
                    print(f'  {rank}. [PROMPT] {diag[:100]}')
                    seen.add(k)
                    rank += 1
                if rank > 9: break
    print()

    # Verdict
    total = sum(bool(r['judge'].get(k, {}).get('pass', False)) for r in results for k, _ in METRIC_KEYS)
    maxv  = n * len(METRIC_KEYS)
    pct   = total / maxv * 100
    print('=' * 72)
    if pct >= 80:
        print(f'LEITURA APROVADA ({total}/{maxv} -- {pct:.0f}%)  Avanca para o proximo capitulo.')
    elif pct >= 60:
        print(f'LEITURA PARCIAL ({total}/{maxv} -- {pct:.0f}%)  Revisar metricas reprovadas.')
    else:
        print(f'LEITURA REPROVADA ({total}/{maxv} -- {pct:.0f}%)  Reescrever antes de continuar.')
    if pagg:
        p_total = sum(r['prompt_judge'][k]['pass'] for r in p_results for k, _ in PROMPT_METRIC_KEYS)
        p_maxv  = pn * len(PROMPT_METRIC_KEYS)
        p_pct   = p_total / p_maxv * 100
        if p_pct >= 75:
            print(f'PROMPT APROVADO ({p_total}/{p_maxv} -- {p_pct:.0f}%)  Prompt gera capacidade util.')
        elif p_pct >= 50:
            print(f'PROMPT PARCIAL ({p_total}/{p_maxv} -- {p_pct:.0f}%)  Refinar prompt antes de publicar.')
        else:
            print(f'PROMPT FRACO ({p_total}/{p_maxv} -- {p_pct:.0f}%)  Prompt precisa ser reescrito.')
    print('=' * 72)
    return agg


if 'EVAL_RESULTS' in dir() and EVAL_RESULTS:
    AGG = render_aggregate(EVAL_RESULTS)
else:
    print('Execute a celula 8 primeiro.')


## Celula 10 -- Gerar patch para a SKILL.md

In [ ]:
# ============================================================
# QUALITY FILTER: remove feedbacks nao aplicaveis antes do patch
# ============================================================

QUALITY_REQUIREMENTS = [
    'sem marcas de ia',
    'dado verificavel',
    'abertura com cena',
    'exercicio com artefato',
    'prompt 5 partes',
    'tensao narrativa',
    'personagem nomeado',
    'coesao com capitulo anterior',
    'insumo anterior usado',
]

def filter_applicable_improvements(improvements, context):
    filtered, rejected = [], []
    irrelevant_patterns = [
        'adicionar mais exemplos genericos',
        'simplificar a linguagem',
        'remover dados',
        'remover o personagem',
        'mais curto',
        'menos exercicio',
        'sem prompt de ia',
    ]
    for imp in improvements:
        il = imp.lower()
        is_irrelevant = any(p in il for p in irrelevant_patterns)
        contradicts_req = any(
            req in il and ('remover' in il or 'eliminar' in il or 'reduzir' in il)
            for req in QUALITY_REQUIREMENTS
        )
        if is_irrelevant or contradicts_req:
            rejected.append(('FILTRADO', imp))
        else:
            filtered.append(imp)
    return filtered, rejected


def generate_skill_patch(results, cap_num, cap_titulo, qg_results=None):
    n = len(results)
    from collections import Counter

    # ── Reading improvements (frequency ranked) ──────────────────────
    read_impr = []
    for r in results:
        read_impr.extend(r['judge'].get('skill_improvements', []))
    freq_r = Counter(imp[:55].lower().strip() for imp in read_impr)
    seen, deduped_read = set(), []
    for imp in sorted(read_impr, key=lambda x: -freq_r[x[:55].lower().strip()]):
        k = imp[:55].lower().strip()
        if k not in seen:
            deduped_read.append((imp, freq_r[k]))
            seen.add(k)

    # ── Prompt improvements ──────────────────────────────────────────
    def _valid_pj(pj):
        if not pj or not isinstance(pj, dict): return False
        return any(k in pj and isinstance(pj.get(k), dict) and 'score' in pj[k]
                   for k, _ in PROMPT_METRIC_KEYS)
    p_results = [r for r in results if _valid_pj(r.get('prompt_judge'))]
    pn = len(p_results)

    prompt_diags = [r['prompt_judge'].get('diagnostico_prompt', '') or ''
                    for r in p_results if isinstance(r.get('prompt_judge'), dict)]
    prompt_suggs = [r['prompt_judge'].get('sugestao_melhoria_prompt', '') or ''
                    for r in p_results if isinstance(r.get('prompt_judge'), dict)]

    # Avg scores per prompt metric
    prompt_scores = {}
    if pn > 0:
        for key, label in PROMPT_METRIC_KEYS:
            scores = [r['prompt_judge'][key]['score']
                      for r in p_results
                      if isinstance(r['prompt_judge'].get(key), dict)]
            if scores:
                prompt_scores[label] = sum(scores) / len(scores)

    weakest_prompt = sorted(prompt_scores.items(), key=lambda x: x[1])[:3] if prompt_scores else []

    # ── Quality Gate context ─────────────────────────────────────────
    qg_context = ''
    if qg_results:
        failed = [k for k, v in qg_results.items() if not v['pass']]
        passed = [k for k, v in qg_results.items() if v['pass']]
        qg_context = (
            f'QUALITY GATE: Aprovados={passed} | Reprovados={failed}\n'
            'REGRA: o patch NAO pode sugerir remover criterios que ja passaram.\n\n'
        )

    # ── Per-persona summary ──────────────────────────────────────────
    def _fmt_persona(r):
        j = r['judge']
        pj = r.get('prompt_judge', {})
        base = (
            f"P{r['persona']['id']} {r['persona']['name']}: "
            f"leit={j.get('leitura_continua',{}).get('score',0)} "
            f"exerc={j.get('exercicio_executado',{}).get('score',0)} "
            f"valor={j.get('valor_percebido',{}).get('score',0)}\n"
            f"  diag: {str(j.get('diagnostico_geral',''))[:200]}"
        )
        if _valid_pj(pj):
            scores = ' '.join(f"{k}={pj[k]['score']}" for k,_ in PROMPT_METRIC_KEYS
                               if isinstance(pj.get(k), dict))
            diag_p = str(pj.get('diagnostico_prompt',''))[:200]
            base += f"\n  prompt: {scores}\n  diag_p: {diag_p}"
        return base
    persona_summary = '\n'.join(_fmt_persona(r) for r in results)

    freq_read_str = '\n'.join(f'  [{cnt}x] {imp}' for imp, cnt in deduped_read[:10])
    weak_prompt_str = '\n'.join(f'  {label}: {avg:.1f}/5 (FRACO)'
                                  for label, avg in weakest_prompt)
    diags_str = '\n'.join(f'  - {d}' for d in prompt_diags if d)
    suggs_str = '\n'.join(f'  - {s}' for s in prompt_suggs if s)

    llm_prompt = (
        f'Voce e o editor-chefe e engenheiro de prompts do livro Profissional Exponencial.\n'
        f'{n} personas avaliaram o capitulo {cap_num}: {cap_titulo}.\n'
        f'{pn} personas executaram o prompt da Capacidade.\n\n'
        f'{qg_context}'
        f'AVALIACOES COMPLETAS POR PERSONA:\n{persona_summary}\n\n'
        f'METRICAS DE LEITURA -- melhorias mais citadas:\n{freq_read_str}\n\n'
        f'METRICAS DO PROMPT -- dimensoes mais fracas:\n{weak_prompt_str}\n\n'
        f'DIAGNOSTICOS DO PROMPT pelas personas:\n{diags_str}\n\n'
        f'SUGESTOES DE MELHORIA DO PROMPT:\n{suggs_str}\n\n'
        'Gere um PATCH com DUAS secoes: LEITURA e PROMPT.\n\n'
        'REGRAS:\n'
        '1. So inclui problemas citados por 2+ personas\n'
        '2. Instrucoes especificas e executaveis sem ambiguidade\n'
        '3. Nao contradiz Quality Gate\n'
        '4. Criterio de sucesso testavel\n'
        '5. SECAO PROMPT e obrigatoria -- o prompt e o produto do livro\n\n'
        f'FORMATO EXATO:\n\n'
        f'=== PATCH -- CAPITULO {cap_num}: {cap_titulo} ===\n'
        f'Personas avaliadas: {n} | Prompt executado por: {pn} personas\n\n'
        '--- SECAO 1: LEITURA ---\n\n'
        'PROBLEMA L1: [nome]\n'
        f'Citado por: [N]/{n} personas\n'
        'Evidencia: [frase exata]\n'
        'Instrucao: [o que mudar no texto do capitulo]\n'
        'Criterio: [como testar que foi resolvido]\n\n'
        '--- SECAO 2: PROMPT DA CAPACIDADE ---\n\n'
        'PROBLEMA P1: [nome da dimensao fraca -- ex: Groundedness, Especificidade]\n'
        f'Score medio: [X.X]/5 em {pn} personas\n'
        'Evidencia: [o que o output gerou de errado]\n'
        'Instrucao: [alteracao especifica no texto do prompt -- qual campo, qual instrucao adicionar]\n'
        'Criterio: [como medir que o prompt melhorou -- ex: groundedness >= 4 em 80% das personas]\n\n'
        'PROBLEMA P2: ...\n\n'
        'REGRA GERAL LEITURA: [se aplicavel a todos os caps]\n'
        'REGRA GERAL PROMPT: [se aplicavel a todos os prompts do livro]\n'
        '=== FIM DO PATCH ==='
    )

    return llm_call(
        'Voce e editor-chefe e engenheiro de prompts especializado em livros de carreira. '
        'Responda em texto estruturado seguindo o formato exato.',
        [], llm_prompt
    )


if 'EVAL_RESULTS' in dir() and EVAL_RESULTS:
    print('Gerando patch filtrado e copiavel...\n')
    qg = QG_RESULTS if 'QG_RESULTS' in dir() else None
    SKILL_PATCH = generate_skill_patch(EVAL_RESULTS, CAPITULO_NUM, CAPITULO_TITULO, qg)

    print(SKILL_PATCH)
    print()
    print('=' * 70)
    print('COMO USAR O PATCH ACIMA:')
    print('  1. Copie tudo entre === PATCH === e === FIM DO PATCH ===')
    print('  2. Cole aqui no Claude junto com o texto do capitulo')
    print('  3. Diga: aplique este patch no capitulo mantendo todos os criterios')
    print('     de qualidade da SKILL.md (Quality Gate, abertura, exercicio, prompt)')
    print('  4. Apos o Claude aplicar, rode o Quality Gate novamente para validar')
    print('  5. Se QG OK, rode as personas de novo para confirmar que melhorou')
    print('=' * 70)
else:
    print('Execute as celulas anteriores primeiro.')


## Celula 11 -- Salvar resultados e patch

In [ ]:
def save_results():
    ts    = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    jname = f'eval_cap{CAPITULO_NUM:02d}_{ts}.json'
    mname = f'skill_patch_cap{CAPITULO_NUM:02d}_{ts}.md'

    output = {
        'timestamp':       ts,
        'capitulo_num':    CAPITULO_NUM,
        'capitulo_titulo': CAPITULO_TITULO,
        'insumos':         INSUMOS_ANTERIORES,
        'palavras':        len(CAPITULO.split()),
        'personas':        len(EVAL_RESULTS),
        'resultados': [
            {
                'id':              r['persona']['id'],
                'nome':            r['persona']['name'],
                'tipo':            r['persona']['type'],
                'reaction':        r['reaction'],
                'judge':           r['judge'],
                'prompt_execution':r.get('prompt_execution', ''),
                'prompt_judge':    r.get('prompt_judge', {}),
            }
            for r in EVAL_RESULTS
        ],
        'skill_patch': SKILL_PATCH if 'SKILL_PATCH' in dir() else 'nao gerado'
    }

    with open(jname, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f'Resultados: {jname}')

    if 'SKILL_PATCH' in dir():
        with open(mname, 'w', encoding='utf-8') as f:
            f.write(f'# Skill Patch -- Cap {CAPITULO_NUM}: {CAPITULO_TITULO}\n')
            f.write(f'Gerado em: {ts}\n\n')
            f.write(SKILL_PATCH)
        print(f'Patch: {mname}')


if 'EVAL_RESULTS' in dir() and EVAL_RESULTS:
    save_results()
else:
    print('Execute as celulas anteriores primeiro.')
